In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [50]:
df= pd.read_csv('Railway_Network_.csv',low_memory=False)
df.shape

(119446, 15)

In [51]:
df.head()

,objectid,id,fromjunction,tojunction,speed,type,noofstations,tracklength,railwayzone,nooflanes,bridge_yn,tunnel_yn,iso_countrycode,remark,st_lengthshape
0,1,80656.0,Tirunelveli Junction,Tenkasi Junction,110.0,Broad Gauge,16.0,72.0,Southern Railway,1.0,N,N,91,NaN,15780.260054
1,2,80668.0,Tirunelveli Junction,Tenkasi Junction,110.0,Broad Gauge,16.0,72.0,Southern Railway,1.0,Y,N,91,NaN,121.012647
2,3,80741.0,Tirunelveli Junction,Tenkasi Junction,110.0,Broad Gauge,16.0,72.0,Southern Railway,1.0,N,N,91,NaN,9891.052902
3,4,80742.0,Tirunelveli Junction,Tenkasi Junction,110.0,Broad Gauge,16.0,72.0,Southern Railway,1.0,Y,N,91,NaN,18.478528
4,5,80743.0,Tirunelveli Junction,Tenkasi Junction,110.0,Broad Gauge,16.0,72.0,Southern Railway,1.0,N,N,91,NaN,708.130380


In [52]:
df.isna().sum()

objectid                0
id                   2104
fromjunction         1004
tojunction           1004
speed               41374
type                  383
noofstations          560
tracklength           560
railwayzone          4248
nooflanes            1468
bridge_yn               1
tunnel_yn             415
iso_countrycode         0
remark             119016
st_lengthshape          0
dtype: int64

In [53]:
df.drop('remark',axis=1,inplace=True)

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119446 entries, 0 to 119445
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   objectid         119446 non-null  int64  
 1   id               117342 non-null  float64
 2   fromjunction     118442 non-null  object 
 3   tojunction       118442 non-null  object 
 4   speed            78072 non-null   float64
 5   type             119063 non-null  object 
 6   noofstations     118886 non-null  float64
 7   tracklength      118886 non-null  float64
 8   railwayzone      115198 non-null  object 
 9   nooflanes        117978 non-null  float64
 10  bridge_yn        119445 non-null  object 
 11  tunnel_yn        119031 non-null  object 
 12  iso_countrycode  119446 non-null  int64  
 13  st_lengthshape   119446 non-null  float64
dtypes: float64(6), int64(2), object(6)
memory usage: 12.8+ MB


In [55]:
df['fromjunction'].nunique()

1570

In [56]:
df = df.dropna(
    subset=["fromjunction", "tojunction", "railwayzone"],
    how="all"
)

In [57]:
df[['fromjunction','tojunction','railwayzone']]

,fromjunction,tojunction,railwayzone
0,Tirunelveli Junction,Tenkasi Junction,Southern Railway
1,Tirunelveli Junction,Tenkasi Junction,Southern Railway
2,Tirunelveli Junction,Tenkasi Junction,Southern Railway
3,Tirunelveli Junction,Tenkasi Junction,Southern Railway
4,Tirunelveli Junction,Tenkasi Junction,Southern Railway
...,...,...,...
119441,Rewa,Baghwar,West Central Railway
119442,Rewa,Baghwar,West Central Railway
119443,Rewa,Baghwar,West Central Railway
119444,Rewa,Baghwar,West Central Railway


In [58]:
pair_zone_check = (
    df.groupby(["fromjunction", "tojunction"])["railwayzone"]
      .nunique(dropna=True)
      .reset_index(name="unique_zones")
)

pair_zone_check["unique_zones"].value_counts()

unique_zones
1    2198
2      59
0      48
Name: count, dtype: int64

In [59]:
pair_zone_map = (
    df.dropna(subset=["fromjunction", "tojunction", "railwayzone"])
      .groupby(["fromjunction", "tojunction"])["railwayzone"]
      .agg(lambda x: x.mode().iloc[0])
      .reset_index()
)

In [60]:
pair_zone_map

,fromjunction,tojunction,railwayzone
0,Abhanpur Junction,Dhamtari,South East Central Railway
1,Abhanpur Junction,Kendri,South East Central Railway
2,Abhanpur Junction,Rajim,South East Central Railway
3,Abhayapuri,Dudhnai Junction,North Frontier Railway
4,Abohar Junction,Bathinda Junction,Northern Railway
...,...,...,...
2252,Yog Nagari Rishikesh,Virbhadra,Northern Railway
2253,Zafarabad Junction,Janghai Junction,Northern Railway
2254,Zafarabad Junction,Jaunpur Junction,Northern Railway
2255,Zafarabad Junction,Sultanpur Junction,Northern Railway


In [61]:
missing_zone = df["railwayzone"].isna()

df_missing_zone = df.loc[
    missing_zone & 
    df["fromjunction"].notna() &
    df["tojunction"].notna()
]

print("Missing zone rows with both junctions:", len(df_missing_zone))

Missing zone rows with both junctions: 4142


In [62]:
mapped_zone = (
    df_missing_zone
    .merge(
        pair_zone_map,
        on=["fromjunction", "tojunction"],
        how="left",
        suffixes=("", "_mapped")
    )
)

print("Can be recovered:", mapped_zone["railwayzone_mapped"].notna().sum())
print("Cannot be recovered:", mapped_zone["railwayzone_mapped"].isna().sum())

Can be recovered: 1560
Cannot be recovered: 2582


In [63]:
conflicting_pairs = pair_zone_check[
    pair_zone_check["unique_zones"] > 1
]

conflicting_pairs

,fromjunction,tojunction,unique_zones
8,Achhnera Junction,Mathura Junction,2
198,Balotra,Samdari Junction,2
219,Bangurkela Outer Cabin,Pakra,2
226,Bansapahar,Manikpur Junction,2
230,Barabanki Junction,Rudauli,2
245,Barddhaman Junction,Saktigarh,2
299,Belsonda,Titlagarh Junction,2
323,Bharatpur Junction,Achhnera Junction,2
326,Bharatpur Outer Cabin,Bandikui Junction,2
348,Bhiwani Junction,Rohtak Junction,2


In [64]:
conflicting_details = (
    df.dropna(subset=["fromjunction", "tojunction", "railwayzone"])
      .groupby(
          ["fromjunction", "tojunction", "railwayzone"]
      )
      .size()
      .reset_index(name="count")
)

conflicting_details = conflicting_details[
    conflicting_details.set_index(
        ["fromjunction", "tojunction"]
    ).index.isin(
        conflicting_pairs.set_index(
            ["fromjunction", "tojunction"]
        ).index
    )
].sort_values(
    ["fromjunction", "tojunction", "count"],
    ascending=[True, True, False]
)

conflicting_details

,fromjunction,tojunction,railwayzone,count
7,Achhnera Junction,Mathura Junction,North Central Railway,50
8,Achhnera Junction,Mathura Junction,West Central Railway,1
187,Balotra,Samdari Junction,North Western Railway,45
188,Balotra,Samdari Junction,South East Central Railway,1
210,Bangurkela Outer Cabin,Pakra,South Eastern Railway,176
...,...,...,...,...
2038,Sohawal,Faizabad Junction,Northern Railway,3
2127,Thana Bihpur Junction,Katihar Junction,East Central Railway,43
2128,Thana Bihpur Junction,Katihar Junction,North Frontier Railway,1
2204,Ujjain C Cabin,Maksi Junction,Western Railway,71


In [65]:
recoverable = mapped_zone[
    mapped_zone["railwayzone_mapped"].notna()
]

print(
    recoverable[
        "railwayzone_mapped"
    ].value_counts()
)

railwayzone_mapped
South Central Railway         848
Western Railway               448
South Eastern Railway         127
Central Railway                72
South East Central Railway     64
West Central Railway            1
Name: count, dtype: int64


In [66]:
print(
    "Recoverable rows:",
    len(recoverable)
)

print(
    "Unique junction pairs:",
    recoverable[
        ["fromjunction", "tojunction"]
    ].drop_duplicates().shape[0]
)

Recoverable rows: 1560
Unique junction pairs: 20


In [67]:
# ---------------------------------------------------------------
# Identify the number of distinct known railway zones for each
# fromjunction -> to junction pair.
#
# unique_zones = 1:
#     The pair consistently belongs to one railway zone.
#
# unique_zones > 1:
#     Conflicting zone information exists, so we do NOT
#     automatically assign a zone.
# ---------------------------------------------------------------

pair_zone_counts = (
    df.dropna(
        subset=["fromjunction", "tojunction", "railwayzone"]
    )
    .groupby(
        ["fromjunction", "tojunction"]
    )["railwayzone"]
    .nunique()
    .reset_index(name="unique_zones")
)

# ---------------------------------------------------------------
# Keep ONLY junction pairs where exactly one railway zone
# is present in the dataset.
# These are our safe mappings.
# ---------------------------------------------------------------

consistent_pairs = pair_zone_counts[
    pair_zone_counts["unique_zones"] == 1
][
    ["fromjunction", "tojunction"]
]

# ---------------------------------------------------------------
# Build the actual zone mapping ONLY from those consistent pairs.
# ---------------------------------------------------------------

safe_zone_mapping = (
    df.dropna(
        subset=["fromjunction", "tojunction", "railwayzone"]
    )
    .merge(
        consistent_pairs,
        on=["fromjunction", "tojunction"],
        how="inner"
    )
    .drop_duplicates(
        subset=["fromjunction", "tojunction", "railwayzone"]
    )
)

# ---------------------------------------------------------------
# Keep only the three columns needed for the mapping.
# ---------------------------------------------------------------

safe_zone_mapping = safe_zone_mapping[
    ["fromjunction", "tojunction", "railwayzone"]
].rename(
    columns={"railwayzone": "railwayzone_mapped"}
)

# ---------------------------------------------------------------
# Merge the safe mapping into the dataset.
# ---------------------------------------------------------------

df = df.merge(
    safe_zone_mapping,
    on=["fromjunction", "tojunction"],
    how="left"
)

# ---------------------------------------------------------------
# Fill ONLY missing railwayzone values for junction pairs
# that have a consistent zone.
#
# Existing railwayzone values are never overwritten.
# Conflicting pairs remain untouched.
# ---------------------------------------------------------------

missing_zone = df["railwayzone"].isna()

fillable = (
    missing_zone &
    df["railwayzone_mapped"].notna()
)

df.loc[fillable, "railwayzone"] = (
    df.loc[fillable, "railwayzone_mapped"]
)

# Remove the temporary mapping column.
df.drop(
    columns=["railwayzone_mapped"],
    inplace=True
)

# ---------------------------------------------------------------
# Verify the result.
# ---------------------------------------------------------------

print(
    "Railwayzone values filled:",
    fillable.sum()
)

print(
    "Remaining missing railwayzone:",
    df["railwayzone"].isna().sum()
)

Railwayzone values filled: 1557
Remaining missing railwayzone: 2585


In [68]:
# ---------------------------------------------------------------
# Count missing values in the three graph-identifying columns.
# These three columns are treated as a related group because
# together they describe a railway connection and its zone.
# ---------------------------------------------------------------

key_columns = [
    "fromjunction",
    "tojunction",
    "railwayzone"
]

print(df[key_columns].isna().sum())

# Show how many records have each combination of missing values.
# This helps us understand which relationships can potentially
# be used for recovery.
print(
    df[key_columns]
    .isna()
    .value_counts()
)

fromjunction     898
tojunction       898
railwayzone     2585
dtype: int64
fromjunction  tojunction  railwayzone
False         False       False          115857
                          True             2585
True          True        False             898
Name: count, dtype: int64


In [69]:
# ---------------------------------------------------------------
# Inspect the 898 records where both graph endpoints are missing.
# Although they cannot be recovered from the junction relationship,
# other attributes may reveal whether these records represent
# useful railway segments or should eventually be excluded.
# ---------------------------------------------------------------

missing_endpoints = df[
    df["fromjunction"].isna() &
    df["tojunction"].isna()
]

print(missing_endpoints.shape)

missing_endpoints[
    [
        "objectid",
        "id",
        "speed",
        "type",
        "noofstations",
        "tracklength",
        "railwayzone",
        "nooflanes",
        "bridge_yn",
        "tunnel_yn",
        "st_lengthshape"
    ]
].head(20)


(898, 14)


,objectid,id,speed,type,noofstations,tracklength,railwayzone,nooflanes,bridge_yn,tunnel_yn,st_lengthshape
813,814,113976.0,NaN,Broad Gauge,NaN,NaN,Southern Railway,NaN,N,N,6207.565998
2390,2391,14337.0,NaN,NaN,NaN,NaN,South East Central Railway,0.0,Y,N,25.008178
3060,3061,99857.0,NaN,Broad Gauge,NaN,NaN,South Western Railway,1.0,Y,N,14.297285
3061,3062,99858.0,NaN,Broad Gauge,NaN,NaN,South Western Railway,1.0,N,N,2811.580563
3062,3063,99859.0,NaN,Broad Gauge,NaN,NaN,South Western Railway,1.0,Y,N,13.117600
3063,3064,99860.0,NaN,Broad Gauge,NaN,NaN,South Western Railway,1.0,N,N,865.955624
3064,3065,99861.0,NaN,Broad Gauge,NaN,NaN,South Western Railway,1.0,Y,N,11.281335
3065,3066,99862.0,NaN,Broad Gauge,NaN,NaN,South Western Railway,1.0,N,N,520.144591
3066,3067,99863.0,NaN,Broad Gauge,NaN,NaN,South Western Railway,1.0,Y,N,18.477326
3067,3068,99864.0,NaN,Broad Gauge,NaN,NaN,South Western Railway,1.0,N,N,423.451589


In [70]:
# ---------------------------------------------------------------
# Check whether records with both junctions missing are
# concentrated in particular railway zones.
# This tells us whether the missing-endpoint problem is
# localized to specific parts of the dataset.
# ---------------------------------------------------------------

missing_endpoints["railwayzone"].value_counts(dropna=False)

railwayzone
South East Central Railway    229
East Coast Railway            105
South Eastern Railway         104
East Central Railway           79
Eastern Railway                78
North Western Railway          76
Central Railway                53
West Central Railway           39
South Central Railway          29
Western Railway                26
North Central Railway          25
South Western Railway          21
                               17
Southern Railway               12
Northern Railway                5
Name: count, dtype: int64

In [71]:
# ---------------------------------------------------------------
# Examine the ID range of records where both endpoints are
# missing. Consecutive IDs may indicate that these records belong
# to contiguous GIS geometry sections.
# ---------------------------------------------------------------

print("Minimum ID:", missing_endpoints["id"].min())
print("Maximum ID:", missing_endpoints["id"].max())

missing_endpoints["id"].sort_values().head(30)

Minimum ID: 0.0
Maximum ID: 114063.0


6802     0.0
7052     0.0
89649    0.0
89650    0.0
89651    0.0
89652    0.0
89653    0.0
89654    0.0
89655    0.0
89630    0.0
89631    0.0
89632    0.0
89633    0.0
89634    0.0
89657    0.0
89658    0.0
89663    0.0
89664    0.0
89643    0.0
89644    0.0
89645    0.0
89646    0.0
89656    0.0
89635    0.0
89636    0.0
89618    0.0
89617    0.0
89616    0.0
89640    0.0
89639    0.0
Name: id, dtype: float64

In [72]:
# ---------------------------------------------------------------
# Display neighbouring records around the first few missing-
# endpoint records.
#
# We want to see whether a record with missing endpoints is
# surrounded by records that contain valid fromjunction and
# tojunction values.
# ---------------------------------------------------------------

for idx in missing_endpoints.index[:5]:
    print("\n--- Missing record index:", idx, "---")
    
    print(
        df.loc[
            max(0, idx - 2): idx + 2,
            [
                "objectid",
                "id",
                "fromjunction",
                "tojunction",
                "type",
                "tracklength",
                "railwayzone",
                "st_lengthshape"
            ]
        ]
    )


--- Missing record index: 813 ---
     objectid        id           fromjunction      tojunction         type  \
811       812  112252.0  Basin Bridge Junction      Korukkupet  Broad Gauge   
812       813  112253.0  Basin Bridge Junction      Korukkupet  Broad Gauge   
813       814  113976.0                    NaN             NaN  Broad Gauge   
814       815  113977.0           Chennai Fort       Velachery  Broad Gauge   
815       816   92401.0         Erode Junction  Salem Junction  Broad Gauge   

     tracklength       railwayzone  st_lengthshape  
811          2.0  Southern Railway       56.629477  
812          2.0  Southern Railway      397.371413  
813          NaN  Southern Railway     6207.565998  
814         19.0  Southern Railway      192.249363  
815         60.0  Southern Railway       14.983913  

--- Missing record index: 2390 ---
      objectid       id        fromjunction             tojunction  \
2388      2389  98406.0  Arakkonam Junction  Basin Bridge Junction

In [73]:
# ---------------------------------------------------------------
# Examine how the 898 records with missing endpoints are
# distributed across railway zones.
#
# If many missing records occur in long consecutive blocks,
# that supports the interpretation that they are groups of
# GIS geometry segments rather than isolated bad records.
# ---------------------------------------------------------------

missing_endpoints_by_zone = (
    missing_endpoints
    .groupby("railwayzone")
    .agg(
        records=("objectid", "count"),
        min_objectid=("objectid", "min"),
        max_objectid=("objectid", "max"),
        total_geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

missing_endpoints_by_zone

,records,min_objectid,max_objectid,total_geometry_length
railwayzone,,,,
South East Central Railway,229,2391,49187,260103.494522
East Coast Railway,105,13851,118297,84918.345291
South Eastern Railway,104,15628,116012,78883.634400
East Central Railway,79,69420,89707,79765.823730
Eastern Railway,78,54575,117689,126453.578615
North Western Railway,76,6803,89665,56661.298053
Central Railway,53,38659,111989,33886.572113
West Central Railway,39,40070,82576,36075.036094
South Central Railway,29,17591,117837,87150.808592


In [74]:
# ---------------------------------------------------------------
# Sort the missing-endpoint records by objectid.
# A new block starts whenever the next objectid is not consecutive.
# ---------------------------------------------------------------

missing_sorted = (
    missing_endpoints
    .sort_values("objectid")
    .copy()
)

missing_sorted["objectid_diff"] = (
    missing_sorted["objectid"].diff()
)

missing_sorted["block"] = (
    missing_sorted["objectid_diff"] != 1
).cumsum()

blocks = (
    missing_sorted
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        railwayzone=("railwayzone", "first"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

blocks.head(20)

,start_objectid,end_objectid,records,railwayzone,geometry_length
block,,,,,
253,89609,89665,57,North Western Railway,26930.563123
338,117952,117990,39,East Coast Railway,18824.994960
339,117992,118023,32,East Coast Railway,8480.276576
25,13853,13873,21,East Coast Railway,16729.500960
159,49171,49187,17,South East Central Railway,18476.937376
3,3061,3077,17,South Western Railway,8229.723768
128,44079,44094,16,South East Central Railway,13514.612429
259,97257,97269,13,Western Railway,13352.410671
139,44329,44340,12,South East Central Railway,7747.965825


In [75]:
# Remove leading/trailing whitespace from railway zone names.
df["railwayzone"] = (
    df["railwayzone"]
    .astype("string")
    .str.strip()
)

# Convert empty strings into proper missing values.
df["railwayzone"] = df["railwayzone"].replace("", pd.NA)

# See the current unique zone names.
print(sorted(df["railwayzone"].dropna().unique()))

['Central Railway', 'East Central Railway', 'East Coast Railwa', 'East Coast Railway', 'Eastern Railway', 'Konkan Railway', 'North Central Railway', 'North Eastern Railway', 'North Frontier Railway', 'North Western Railway', 'Northeast Frontier Railway', 'Northern Railway', 'South Central Railway', 'South East Central Railway', 'South Eastern Railway', 'South Western Railway', 'Southen Railway', 'Southern Railway', 'West Central Railway', 'Westem Railway', 'Western Railway']


In [76]:
# Count records belonging to each zone.
zone_counts = df["railwayzone"].value_counts(dropna=False)

print(zone_counts)

railwayzone
South Central Railway         15090
Western Railway               14209
South Western Railway          8475
Southern Railway               8011
North Western Railway          7768
Northern Railway               6862
West Central Railway           6830
East Coast Railway             6632
East Central Railway           6386
Central Railway                6090
North Central Railway          6035
North Frontier Railway         5906
South Eastern Railway          4921
South East Central Railway     4525
North Eastern Railway          3715
Eastern Railway                3521
<NA>                           2602
Konkan Railway                 1426
Northeast Frontier Railway      326
Southen Railway                   5
East Coast Railwa                 4
Westem Railway                    1
Name: count, dtype: Int64


In [77]:
df[(df['railwayzone']=='North Frontier Railway') | (df['railwayzone']=='Northeast Frontier Railway')]

,objectid,id,fromjunction,tojunction,speed,type,noofstations,tracklength,railwayzone,nooflanes,bridge_yn,tunnel_yn,iso_countrycode,st_lengthshape
49203,49204,43289.0,Lumding Junction,Badarpur Junction,NaN,Broad Gauge,33.0,169.0,North Frontier Railway,1.0,N,N,91,3139.295969
49204,49205,43290.0,Lumding Junction,Badarpur Junction,NaN,Broad Gauge,33.0,169.0,North Frontier Railway,1.0,Y,N,91,26.868092
49205,49206,43298.0,Lumding Junction,Badarpur Junction,NaN,Broad Gauge,33.0,169.0,North Frontier Railway,1.0,Y,N,91,28.350450
49206,49207,43299.0,Lumding Junction,Badarpur Junction,NaN,Broad Gauge,33.0,169.0,North Frontier Railway,1.0,N,N,91,423.125768
49207,49208,43300.0,Lumding Junction,Badarpur Junction,NaN,Broad Gauge,33.0,169.0,North Frontier Railway,1.0,N,N,91,2385.471561
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119113,119220,NaN,Agartala,Gangasagar Bangladesh,NaN,Broad Gauge,4.0,6.0,Northeast Frontier Railway,NaN,Y,N,91,3290.934978
119114,119221,NaN,Kamalajari,Byrnihat,NaN,Broad Gauge,8.0,9.0,Northeast Frontier Railway,NaN,Y,N,91,85.937944
119115,119222,NaN,Kamalajari,Byrnihat,NaN,Broad Gauge,8.0,9.0,Northeast Frontier Railway,NaN,N,N,91,1114.170759
119116,119223,NaN,Pawakhali,Thakurganj,NaN,Broad Gauge,14.0,22.0,Northeast Frontier Railway,NaN,Y,N,91,187.292536


In [78]:
# ---------------------------------------------------------------
# Standardize the known variant of Northeast Frontier Railway.
# The reference dataset uses "Northeast Frontier Railway" as
# the canonical railway-zone name.
# ---------------------------------------------------------------

df["railwayzone"] = df["railwayzone"].replace(
    {
        "North Frontier Railway": "Northeast Frontier Railway"
    }
)

In [79]:
print(
    df["railwayzone"].value_counts()
    .loc[lambda x: x.index.str.contains(
        "Frontier", na=False
    )]
)

railwayzone
Northeast Frontier Railway    6232
Name: count, dtype: Int64


In [80]:
# ---------------------------------------------------------------
# Correct obvious spelling/typing errors in railway zone names.
#
# These replacements do not change the identity of a railway zone;
# they only standardize misspelled variants to the canonical name.
# ---------------------------------------------------------------

zone_spelling_corrections = {
    "Southen Railway": "Southern Railway",
    "East Coast Railwa": "East Coast Railway",
    "Westem Railway": "Western Railway"
}

df["railwayzone"] = df["railwayzone"].replace(
    zone_spelling_corrections
)

In [81]:
# Check the standardized railway-zone names.
print(
    df["railwayzone"].value_counts(dropna=False)
)

railwayzone
South Central Railway         15090
Western Railway               14210
South Western Railway          8475
Southern Railway               8016
North Western Railway          7768
Northern Railway               6862
West Central Railway           6830
East Coast Railway             6636
East Central Railway           6386
Northeast Frontier Railway     6232
Central Railway                6090
North Central Railway          6035
South Eastern Railway          4921
South East Central Railway     4525
North Eastern Railway          3715
Eastern Railway                3521
<NA>                           2602
Konkan Railway                 1426
Name: count, dtype: Int64


In [82]:
# These should return 0 for each spelling variant.
for old_name in zone_spelling_corrections:
    print(
        old_name,
        "→",
        (df["railwayzone"] == old_name).sum()
    )

Southen Railway → 0
East Coast Railwa → 0
Westem Railway → 0


In [83]:
# ---------------------------------------------------------------
# Count records with both junctions missing for each railway zone.
# This gives us the baseline for zone-wise recovery analysis.
# ---------------------------------------------------------------

missing_junctions = df[
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

missing_by_zone = (
    missing_junctions["railwayzone"]
    .value_counts(dropna=False)
)

print(missing_by_zone)

railwayzone
South East Central Railway    229
East Coast Railway            105
South Eastern Railway         104
East Central Railway           79
Eastern Railway                78
North Western Railway          76
Central Railway                53
West Central Railway           39
South Central Railway          29
Western Railway                26
North Central Railway          25
South Western Railway          21
<NA>                           17
Southern Railway               12
Northern Railway                5
Name: count, dtype: Int64


In [84]:
# ---------------------------------------------------------------
# Inspect the other attributes of records where both junctions
# are missing, starting with South East Central Railway.
#
# The purpose is to identify whether these records contain
# recurring feature patterns that can later be matched against
# records where the junctions are known.
# ---------------------------------------------------------------

sec_missing = missing_junctions[
    missing_junctions["railwayzone"] == "South East Central Railway"
].copy()

print("Number of records:", len(sec_missing))

# Examine the categorical attributes.
print("\nTYPE:")
print(sec_missing["type"].value_counts(dropna=False))

print("\nNO. OF LANES:")
print(sec_missing["nooflanes"].value_counts(dropna=False))

print("\nBRIDGE:")
print(sec_missing["bridge_yn"].value_counts(dropna=False))

print("\nTUNNEL:")
print(sec_missing["tunnel_yn"].value_counts(dropna=False))

Number of records: 229

TYPE:
type
NaN            228
Broad Gauge      1
Name: count, dtype: int64

NO. OF LANES:
nooflanes
0.0    227
NaN      2
Name: count, dtype: int64

BRIDGE:
bridge_yn
N    126
Y    103
Name: count, dtype: int64

TUNNEL:
tunnel_yn
N    229
Name: count, dtype: int64


In [85]:
# ---------------------------------------------------------------
# Numerical feature distribution for the same 229 records.
# ---------------------------------------------------------------

print(
    sec_missing[
        [
            "speed",
            "noofstations",
            "tracklength",
            "nooflanes",
            "st_lengthshape"
        ]
    ].describe()
)

       speed  noofstations  tracklength  nooflanes  st_lengthshape
count    0.0           0.0          0.0      227.0      229.000000
mean     NaN           NaN          NaN        0.0     1135.823120
std      NaN           NaN          NaN        0.0     2415.985627
min      NaN           NaN          NaN        0.0        9.113557
25%      NaN           NaN          NaN        0.0       32.526638
50%      NaN           NaN          NaN        0.0      226.422477
75%      NaN           NaN          NaN        0.0     1190.718617
max      NaN           NaN          NaN        0.0    20146.496891


In [86]:
# ---------------------------------------------------------------
# Find the most frequent known junction pairs in South East
# Central Railway, separated by bridge status.
#
# This lets us see whether bridge status provides any useful
# information about which junction pairs may correspond to the
# missing records.
# ---------------------------------------------------------------

sec_complete = df[
    (df["railwayzone"] == "South East Central Railway") &
    df["fromjunction"].notna() &
    df["tojunction"].notna()
].copy()

for bridge_status in ["Y", "N"]:
    
    print(f"\n--- Bridge = {bridge_status} ---")
    
    pairs = (
        sec_complete[
            sec_complete["bridge_yn"] == bridge_status
        ]
        .groupby(
            ["fromjunction", "tojunction"]
        )
        .size()
        .sort_values(ascending=False)
        .head(20)
    )
    
    print(pairs)


--- Bridge = Y ---
fromjunction          tojunction           
Chhindwara Junction   Nainpur Junction         236
Nainpur Junction      Kachhpura Junction       177
Itwari Junction       Chhindwara Junction      171
Boridand Junction     Ambikapur                169
Balaghat Junction     Nainpur Junction         137
                      Tumsar Road Junction     116
Bilaspur Chord Cabin  Pendra Road              108
Singhpur              Rupaund                  105
Dallirajhara          Antagarh\n                93
Urkura                Bilaspur Bypass Cabin     91
Mandla Fort           Nainpur Junction          71
Marauda               Dallirajhara              60
Anuppur Junction      Boridand Junction         49
Balaghat Junction     Gondia Junction           46
Gondia Junction       Darekasa                  42
Robertson             Himgir                    39
Abhanpur Junction     Dhamtari                  37
Paniajob              Raj Nandgaon              32
Sarona            

In [87]:
# ---------------------------------------------------------------
# Compare geometry-length patterns of known junction pairs.
# We will use this to determine whether st_lengthshape can
# potentially help identify the missing junction pairs.
# ---------------------------------------------------------------

sec_pair_lengths = (
    sec_complete
    .groupby(["fromjunction", "tojunction"])
    ["st_lengthshape"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
    .reset_index()
)

sec_pair_lengths.sort_values(
    "count",
    ascending=False
).head(20)

,fromjunction,tojunction,count,mean,median,min,max
25,Chhindwara Junction,Nainpur Junction,476,317.235928,83.822753,7.364571,3916.191593
40,Itwari Junction,Chhindwara Junction,361,285.063686,101.746253,5.263939,2585.770990
53,Nainpur Junction,Kachhpura Junction,355,345.081056,110.187979,8.082398,4381.727869
19,Boridand Junction,Ambikapur,342,368.716191,77.431887,5.447775,4362.660155
10,Balaghat Junction,Nainpur Junction,276,293.441171,90.008225,9.533456,2330.942653
11,Balaghat Junction,Tumsar Road Junction,235,373.246533,94.561646,5.062384,2835.668786
16,Bilaspur Chord Cabin,Pendra Road,219,487.126155,101.858553,7.451204,8457.299945
68,Singhpur,Rupaund,211,566.723882,114.327633,5.102744,5904.778306
27,Dallirajhara,Antagarh\n,187,334.585907,83.700701,6.668160,2146.696384
72,Urkura,Bilaspur Bypass Cabin,186,593.414803,129.957810,15.111911,7392.609567


In [88]:
# Geometry-length distribution of the 229 missing-junction records.

sec_missing["st_lengthshape"].describe()

count      229.000000
mean      1135.823120
std       2415.985627
min          9.113557
25%         32.526638
50%        226.422477
75%       1190.718617
max      20146.496891
Name: st_lengthshape, dtype: float64

In [89]:
# Check whether geometry lengths are repeated in the complete
# South East Central Railway records.
length_counts = (
    sec_complete["st_lengthshape"]
    .round(3)
    .value_counts()
)

print(length_counts.head(20))

st_lengthshape
34.171     2
14.483     2
27.246     2
26.012     2
16.585     2
26.059     2
28.411     2
28.166     2
566.095    2
28.765     2
9.971      2
15.462     2
27.776     2
21.568     2
48.236     2
31.118     2
19.334     2
15.394     2
19.342     2
11.447     2
Name: count, dtype: int64


In [90]:
sec_complete

,objectid,id,fromjunction,tojunction,speed,type,noofstations,tracklength,railwayzone,nooflanes,bridge_yn,tunnel_yn,iso_countrycode,st_lengthshape
13957,13958,11672.0,Sambalpur Road,Sarla Junction,NaN,Broad Gauge,3.0,5.0,South East Central Railway,2.0,Y,N,91,15.197949
13985,13986,14133.0,Himgir,Jharsuguda Bypass Cabin,NaN,Broad Gauge,21.0,32.0,South East Central Railway,3.0,Y,N,91,17.334904
13986,13987,14141.0,Himgir,Jharsuguda Bypass Cabin,NaN,Broad Gauge,21.0,32.0,South East Central Railway,3.0,N,N,91,981.684223
13987,13988,14154.0,Himgir,Jharsuguda Bypass Cabin,NaN,Broad Gauge,21.0,32.0,South East Central Railway,3.0,Y,N,91,462.610404
13988,13989,14182.0,Himgir,Jharsuguda Bypass Cabin,NaN,Broad Gauge,21.0,32.0,South East Central Railway,3.0,N,N,91,1168.785353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119144,119251,NaN,Antagarh,Taroki,NaN,Broad Gauge,20.0,17.0,South East Central Railway,NaN,N,N,91,1271.058564
119145,119252,NaN,Antagarh,Taroki,NaN,Broad Gauge,20.0,17.0,South East Central Railway,NaN,Y,N,91,8.841443
119146,119253,NaN,Antagarh,Taroki,NaN,Broad Gauge,20.0,17.0,South East Central Railway,NaN,N,N,91,150.056813
119147,119254,NaN,Taroki,Jagdalpur,NaN,Broad Gauge,20.0,153.0,South East Central Railway,NaN,N,N,91,19726.144449


In [94]:
# ---------------------------------------------------------------
# Create the same rounded geometry-length key for all datasets
# ---------------------------------------------------------------

sec_complete["length_key"] = (
    sec_complete["st_lengthshape"].round(3)
)

sec_missing["length_key"] = (
    sec_missing["st_lengthshape"].round(3)
)

pattern_mapping["length_key"] = (
    pattern_mapping["st_lengthshape"].round(3)
)

In [95]:
# ---------------------------------------------------------------
# Check whether the available features can uniquely identify
# junction pairs in South East Central Railway.
#
# We round st_lengthshape to 3 decimals because tiny floating-point
# differences should not be treated as different geometry lengths.
# ---------------------------------------------------------------

# Create one identifier for each known junction pair.
sec_complete["junction_pair"] = (
    sec_complete["fromjunction"].astype(str)
    + " → "
    + sec_complete["tojunction"].astype(str)
)

# Count how many different junction pairs share the same
# bridge status + geometry length.
pattern_mapping = (
    sec_complete
    .groupby(["bridge_yn", "length_key"])["junction_pair"]
    .nunique()
    .reset_index(name="unique_junction_pairs")
)

print(pattern_mapping["unique_junction_pairs"].value_counts())

unique_junction_pairs
1    4244
2      24
Name: count, dtype: int64


In [96]:
print("sec_missing:", sec_missing.shape)
print("pattern_mapping:", pattern_mapping.shape)
print("sec_complete:", sec_complete.shape)

print("\nsec_missing columns:")
print(sec_missing.columns.tolist())

print("\npattern_mapping columns:")
print(pattern_mapping.columns.tolist())

print("\nsec_complete columns:")
print(sec_complete.columns.tolist())

sec_missing: (229, 15)
pattern_mapping: (4268, 3)
sec_complete: (4296, 16)

sec_missing columns:
['objectid', 'id', 'fromjunction', 'tojunction', 'speed', 'type', 'noofstations', 'tracklength', 'railwayzone', 'nooflanes', 'bridge_yn', 'tunnel_yn', 'iso_countrycode', 'st_lengthshape', 'length_key']

pattern_mapping columns:
['bridge_yn', 'length_key', 'unique_junction_pairs']

sec_complete columns:
['objectid', 'id', 'fromjunction', 'tojunction', 'speed', 'type', 'noofstations', 'tracklength', 'railwayzone', 'nooflanes', 'bridge_yn', 'tunnel_yn', 'iso_countrycode', 'st_lengthshape', 'junction_pair', 'length_key']


In [97]:
# ---------------------------------------------------------------
# Test whether the available feature combination
# (bridge_yn + st_lengthshape) can recover any of the
# 229 missing junction records.
#
# We use the same rounding precision as the complete records.
# ---------------------------------------------------------------

sec_missing["length_key"] = (
    sec_missing["st_lengthshape"].round(3)
)

# Create a mapping from:
# bridge status + geometry length
# -> known junction pair
#
# Only use patterns that correspond to exactly ONE junction pair.
safe_patterns = (
    pattern_mapping[
        pattern_mapping["unique_junction_pairs"] == 1
    ][
        ["bridge_yn", "length_key"]
    ]
)

# Attach the known junction pair to the safe patterns.
safe_pattern_mapping = (
    sec_complete[
        ["bridge_yn", "length_key", "junction_pair"]
    ]
    .drop_duplicates(
        ["bridge_yn", "length_key"]
    )
    .merge(
        safe_patterns,
        on=["bridge_yn", "length_key"],
        how="inner"
    )
)

# Match the missing records against the safe patterns.
sec_recovery_test = sec_missing.merge(
    safe_pattern_mapping,
    on=["bridge_yn", "length_key"],
    how="left"
)

print(
    "Potentially recoverable:",
    sec_recovery_test["junction_pair"].notna().sum()
)

print(
    "No matching pattern:",
    sec_recovery_test["junction_pair"].isna().sum()
)

Potentially recoverable: 3
No matching pattern: 226


In [98]:
# ---------------------------------------------------------------
# Display the three records whose bridge status + geometry length
# uniquely matched an existing junction pair.
#
# We inspect them before filling to verify that the matches
# are sensible and not accidental.
# ---------------------------------------------------------------

sec_recovery_test[
    sec_recovery_test["junction_pair"].notna()
][
    [
        "objectid",
        "bridge_yn",
        "st_lengthshape",
        "junction_pair"
    ]
]

,objectid,bridge_yn,st_lengthshape,junction_pair
46,40525,Y,32.545274,Nainpur Junction → Kachhpura Junction
57,40971,Y,23.534212,Sarona → Urkura
131,44017,Y,32.922955,Pendra Road → Chhulha


In [99]:
# ---------------------------------------------------------------
# Recover the 3 SEC Railway records whose combination of
# bridge_yn + rounded st_lengthshape uniquely identified
# a known junction pair.
#
# Only these verified matches are filled.
# ---------------------------------------------------------------

# Extract only the verified matches.
verified_matches = sec_recovery_test[
    sec_recovery_test["junction_pair"].notna()
].copy()

# Split the identified junction pair back into fromjunction
# and tojunction.
verified_matches[
    ["recovered_fromjunction", "recovered_tojunction"]
] = verified_matches["junction_pair"].str.split(
    " → ",
    expand=True
)

# Use objectid to update the original dataframe.
for _, row in verified_matches.iterrows():

    df.loc[
        df["objectid"] == row["objectid"],
        "fromjunction"
    ] = row["recovered_fromjunction"]

    df.loc[
        df["objectid"] == row["objectid"],
        "tojunction"
    ] = row["recovered_tojunction"]

print("Records recovered:", len(verified_matches))

Records recovered: 3


In [100]:
df['railwayzone'].unique()

<StringArray>
[          'Southern Railway',           'Northern Railway',
            'Western Railway',      'South Western Railway',
 'South East Central Railway',      'South Central Railway',
      'North Central Railway',      'South Eastern Railway',
            'Eastern Railway',      'North Western Railway',
         'East Coast Railway',            'Central Railway',
             'Konkan Railway',                         <NA>,
      'North Eastern Railway',       'East Central Railway',
       'West Central Railway', 'Northeast Frontier Railway']
Length: 18, dtype: string

In [101]:
# Verify the remaining missing-junction records.
print(
    df[
        df["fromjunction"].isna() &
        df["tojunction"].isna() &
        (df['railwayzone']=='South East Central Railway')
    ].shape[0]
)

226


In [102]:
# ---------------------------------------------------------------
# Test whether adding "type" to the previous feature combination
# provides additional unique junction-pair matches.
#
# We only use records where all three features are available.
# A pattern is considered safe only when it maps to exactly one
# known junction pair.
# ---------------------------------------------------------------

# Remaining SEC records with both junctions still missing
sec_remaining = df[
    (df["railwayzone"] == "South East Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

# Round geometry length to the same precision used previously.
sec_remaining["length_key"] = (
    sec_remaining["st_lengthshape"].round(3)
)

# Prepare complete SEC records
sec_complete["length_key"] = (
    sec_complete["st_lengthshape"].round(3)
)

# Build the feature-pattern -> junction-pair relationship.
type_pattern = (
    sec_complete
    .dropna(
        subset=[
            "bridge_yn",
            "length_key",
            "type"
        ]
    )
    .groupby(
        ["bridge_yn", "length_key", "type"]
    )["junction_pair"]
    .nunique()
    .reset_index(name="unique_junction_pairs")
)

print(
    type_pattern["unique_junction_pairs"].value_counts()
)

# Keep only patterns that identify exactly one junction pair.
safe_type_patterns = type_pattern[
    type_pattern["unique_junction_pairs"] == 1
]

# Create the actual mapping.
type_mapping = (
    sec_complete
    .dropna(
        subset=[
            "bridge_yn",
            "length_key",
            "type"
        ]
    )
    [
        [
            "bridge_yn",
            "length_key",
            "type",
            "junction_pair"
        ]
    ]
    .drop_duplicates(
        [
            "bridge_yn",
            "length_key",
            "type"
        ]
    )
    .merge(
        safe_type_patterns[
            [
                "bridge_yn",
                "length_key",
                "type"
            ]
        ],
        on=[
            "bridge_yn",
            "length_key",
            "type"
        ],
        how="inner"
    )
)

# Test the remaining missing records.
type_recovery_test = sec_remaining.merge(
    type_mapping,
    on=[
        "bridge_yn",
        "length_key",
        "type"
    ],
    how="left"
)

print(
    "Potentially recoverable:",
    type_recovery_test["junction_pair"].notna().sum()
)

print(
    "No matching pattern:",
    type_recovery_test["junction_pair"].isna().sum()
)

unique_junction_pairs
1    4246
2      23
Name: count, dtype: int64
Potentially recoverable: 0
No matching pattern: 226


In [103]:
# ---------------------------------------------------------------
# Divide st_lengthshape into length ranges.
#
# The purpose is to check whether records within a similar
# geometry-length range tend to belong to a small number of
# junction pairs.
# ---------------------------------------------------------------

sec_complete["length_bin"] = pd.cut(
    sec_complete["st_lengthshape"],
    bins=[0, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000, float("inf")]
)

length_bin_pairs = (
    sec_complete
    .groupby("length_bin", observed=True)["junction_pair"]
    .nunique()
    .reset_index(name="unique_junction_pairs")
)

print(length_bin_pairs)

          length_bin  unique_junction_pairs
0        (0.0, 25.0]                     51
1       (25.0, 50.0]                     53
2      (50.0, 100.0]                     40
3     (100.0, 250.0]                     44
4     (250.0, 500.0]                     46
5    (500.0, 1000.0]                     51
6   (1000.0, 2500.0]                     58
7   (2500.0, 5000.0]                     43
8  (5000.0, 10000.0]                     25
9     (10000.0, inf]                      7


In [104]:
# ---------------------------------------------------------------
# Test whether the combination of ALL available attributes can
# uniquely identify a junction pair.
#
# We use the complete SEC Railway records as the reference.
# A feature combination is considered safe only when it maps
# to exactly ONE known junction pair.
# ---------------------------------------------------------------

# Features that are actually useful/available in the
# remaining missing-junction records.
features = [
    "bridge_yn",
    "tunnel_yn",
    "nooflanes",
    "st_lengthshape"
]

# Create rounded length key so tiny floating-point differences
# don't prevent otherwise identical geometry lengths from matching.
sec_complete["length_key"] = (
    sec_complete["st_lengthshape"].round(3)
)

sec_remaining["length_key"] = (
    sec_remaining["st_lengthshape"].round(3)
)

# Replace the original continuous length with the rounded key
# for the pattern matching.
match_features = [
    "bridge_yn",
    "tunnel_yn",
    "nooflanes",
    "length_key"
]

# ---------------------------------------------------------------
# Determine how many different junction pairs correspond to
# each complete feature combination.
# ---------------------------------------------------------------

feature_pattern_counts = (
    sec_complete
    .dropna(subset=match_features)
    .groupby(match_features)["junction_pair"]
    .nunique()
    .reset_index(name="unique_junction_pairs")
)

print(
    feature_pattern_counts["unique_junction_pairs"]
    .value_counts()
)

# ---------------------------------------------------------------
# Keep only feature combinations that identify exactly ONE
# junction pair.
# ---------------------------------------------------------------

safe_patterns = feature_pattern_counts[
    feature_pattern_counts["unique_junction_pairs"] == 1
]

# Build the actual feature -> junction-pair mapping.
feature_mapping = (
    sec_complete
    .dropna(
        subset=match_features + ["junction_pair"]
    )
    [
        match_features + ["junction_pair"]
    ]
    .drop_duplicates()
    .merge(
        safe_patterns[match_features],
        on=match_features,
        how="inner"
    )
)

# ---------------------------------------------------------------
# Test the 226 remaining missing-junction records.
# ---------------------------------------------------------------

full_feature_test = sec_remaining.merge(
    feature_mapping,
    on=match_features,
    how="left"
)

print(
    "Potentially recoverable:",
    full_feature_test["junction_pair"].notna().sum()
)

print(
    "No matching pattern:",
    full_feature_test["junction_pair"].isna().sum()
)

unique_junction_pairs
1    4229
2      10
Name: count, dtype: int64
Potentially recoverable: 0
No matching pattern: 226


In [105]:
# ---------------------------------------------------------------
# Create geometry-length bins for the complete SEC records.
# This allows records with similar, rather than identical,
# geometry lengths to be compared.
# ---------------------------------------------------------------

bins = [
    0, 25, 50, 100, 250, 500,
    1000, 2500, 5000, 10000, float("inf")
]

labels = [
    "0-25",
    "25-50",
    "50-100",
    "100-250",
    "250-500",
    "500-1000",
    "1000-2500",
    "2500-5000",
    "5000-10000",
    "10000+"
]

sec_complete["length_bin"] = pd.cut(
    sec_complete["st_lengthshape"],
    bins=bins,
    labels=labels
)

sec_remaining["length_bin"] = pd.cut(
    sec_remaining["st_lengthshape"],
    bins=bins,
    labels=labels
)

In [106]:
# ---------------------------------------------------------------
# Test the combined categorical/infrastructure pattern:
#
# bridge + tunnel + lanes + geometry-length range
#
# We check whether this combination narrows the possible
# junction pairs sufficiently.
# ---------------------------------------------------------------

match_features = [
    "bridge_yn",
    "tunnel_yn",
    "nooflanes",
    "length_bin"
]

pattern_counts = (
    sec_complete
    .dropna(subset=match_features)
    .groupby(match_features, observed=True)["junction_pair"]
    .nunique()
    .reset_index(name="unique_junction_pairs")
)

print(
    pattern_counts["unique_junction_pairs"]
    .value_counts()
)

unique_junction_pairs
1     16
3      6
2      5
7      4
9      4
10     3
8      3
4      3
16     2
15     2
12     2
18     1
17     1
23     1
24     1
21     1
14     1
5      1
6      1
11     1
19     1
25     1
22     1
13     1
Name: count, dtype: int64


In [107]:
# ---------------------------------------------------------------
# Keep only patterns that correspond to exactly one known
# junction pair.
# ---------------------------------------------------------------

safe_patterns = pattern_counts[
    pattern_counts["unique_junction_pairs"] == 1
]

pattern_mapping = (
    sec_complete
    .dropna(
        subset=match_features + ["junction_pair"]
    )
    [
        match_features + ["junction_pair"]
    ]
    .drop_duplicates()
    .merge(
        safe_patterns[match_features],
        on=match_features,
        how="inner"
    )
)

# ---------------------------------------------------------------
# Test the 226 missing-junction records.
# ---------------------------------------------------------------

range_test = sec_remaining.merge(
    pattern_mapping,
    on=match_features,
    how="left"
)

print(
    "Potentially recoverable:",
    range_test["junction_pair"].notna().sum()
)

print(
    "No matching pattern:",
    range_test["junction_pair"].isna().sum()
)

Potentially recoverable: 32
No matching pattern: 194


In [108]:
# ---------------------------------------------------------------
# Inspect the 32 potential matches before recovery.
#
# We want to verify which missing records matched which junction
# pairs and what feature pattern produced the match.
# ---------------------------------------------------------------

recoverable_32 = range_test[
    range_test["junction_pair"].notna()
][
    [
        "objectid",
        "bridge_yn",
        "tunnel_yn",
        "nooflanes",
        "st_lengthshape",
        "length_bin",
        "junction_pair"
    ]
].copy()

print(recoverable_32)

     objectid bridge_yn tunnel_yn  nooflanes  st_lengthshape  length_bin  \
11      40466         N         N        0.0     4645.349815   2500-5000   
39      40513         N         N        0.0     2952.710783   2500-5000   
50      40713         N         N        0.0     2805.853116   2500-5000   
53      40961         N         N        0.0     7149.437031  5000-10000   
59      41562         N         N        0.0     6056.334888  5000-10000   
83      42104         N         N        0.0     2880.104133   2500-5000   
86      42114         N         N        0.0     4330.521768   2500-5000   
90      42530         N         N        0.0     6306.845766  5000-10000   
91      42531         N         N        0.0     9004.103921  5000-10000   
94      42535         N         N        0.0     9591.887718  5000-10000   
95      42536         N         N        0.0     7996.534063  5000-10000   
96      42537         Y         N        0.0      110.647534     100-250   
100     4265

In [109]:
print(
    "Unique recovered junction pairs:",
    recoverable_32["junction_pair"].nunique()
)

Unique recovered junction pairs: 1


In [110]:
# ---------------------------------------------------------------
# Recover the 32 SEC Railway records that uniquely match the
# feature pattern of:
#
# Gavera Road → Pendra Road
#
# All 32 records map to the same known junction pair, so this
# is treated as a reliable recovery.
# ---------------------------------------------------------------

recoverable_32 = range_test[
    range_test["junction_pair"].notna()
].copy()

# Split the known junction pair into its two endpoints.
recoverable_32[
    ["recovered_fromjunction", "recovered_tojunction"]
] = recoverable_32["junction_pair"].str.split(
    " → ",
    expand=True
)

# Update the original dataframe using objectid.
for _, row in recoverable_32.iterrows():

    df.loc[
        df["objectid"] == row["objectid"],
        "fromjunction"
    ] = row["recovered_fromjunction"]

    df.loc[
        df["objectid"] == row["objectid"],
        "tojunction"
    ] = row["recovered_tojunction"]

print("Records recovered:", len(recoverable_32))

Records recovered: 32


In [111]:
# Check how many SEC records still have both junctions missing.
sec_remaining_count = (
    (df["railwayzone"] == "South East Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print(
    "Remaining SEC missing-junction records:",
    sec_remaining_count
)

Remaining SEC missing-junction records: 194


In [112]:
# ---------------------------------------------------------------
# Check whether tracklength and noofstations contain usable
# information in the remaining SEC missing-junction records.
# ---------------------------------------------------------------

sec_remaining = df[
    (df["railwayzone"] == "South East Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

print("Remaining records:", len(sec_remaining))

print("\nTracklength:")
print(sec_remaining["tracklength"].notna().sum())

print("\nNo. of stations:")
print(sec_remaining["noofstations"].notna().sum())

print("\nSpeed:")
print(sec_remaining["speed"].notna().sum())

print("\nType:")
print(sec_remaining["type"].notna().sum())

Remaining records: 194

Tracklength:
0

No. of stations:
0

Speed:
0

Type:
1


In [113]:
# ---------------------------------------------------------------
# Find contiguous blocks of the remaining SEC records whose
# fromjunction and to junction are both missing.
# ---------------------------------------------------------------

sec_remaining = df[
    (df["railwayzone"] == "South East Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

sec_remaining = sec_remaining.sort_values("objectid")

sec_remaining["objectid_diff"] = (
    sec_remaining["objectid"].diff()
)

sec_remaining["block"] = (
    sec_remaining["objectid_diff"] != 1
).cumsum()

sec_blocks = (
    sec_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(sec_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
74              44329         44340       12      7747.965825
65              44086         44094        9      4095.648642
13              40488         40495        8      3764.709375
71              44127         44134        8      4155.468616
67              44105         44110        6      2294.994908
...               ...           ...      ...              ...
61              44065         44065        1      1103.625622
51              43966         43966        1        18.905868
55              44016         44016        1       869.591826
56              44018         44018        1       436.160940
59              44053         44053        1      1318.589953

[79 rows x 4 columns]


In [114]:
# ---------------------------------------------------------------
# For every contiguous block of missing SEC junctions, inspect
# the records immediately before and after the block.
#
# If both neighbouring records have the same fromjunction and
# tojunction, the missing block may belong to that same junction
# pair.
# ---------------------------------------------------------------

# Get the objectids marking the beginning and end of each block.
block_ranges = (
    sec_blocks.reset_index()
    [["block", "start_objectid", "end_objectid", "records"]]
)

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]

        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = (
                f"{b['fromjunction']} → {b['tojunction']}"
            )

    if len(after) > 0:
        a = after.iloc[0]

        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = (
                f"{a['fromjunction']} → {a['tojunction']}"
            )

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

neighbour_check = pd.DataFrame(neighbour_results)

print(neighbour_check)


    block  start_objectid  end_objectid  records  \
0      74           44329         44340       12   
1      65           44086         44094        9   
2      13           40488         40495        8   
3      71           44127         44134        8   
4      67           44105         44110        6   
..    ...             ...           ...      ...   
74     61           44065         44065        1   
75     51           43966         43966        1   
76     55           44016         44016        1   
77     56           44018         44018        1   
78     59           44053         44053        1   

                       before_pair                      after_pair  
0    Boridand Junction → Ambikapur        Gevra Road → Pendra Road  
1        Gavera Road → Pendra Road   Boridand Junction → Ambikapur  
2   Urkura → Bilaspur Bypass Cabin  Urkura → Bilaspur Bypass Cabin  
3    Boridand Junction → Ambikapur   Boridand Junction → Ambikapur  
4    Boridand Junction → Ambik

In [115]:
# ---------------------------------------------------------------
# Identify blocks where the immediately preceding and following
# records both belong to the same known junction pair.
# ---------------------------------------------------------------

same_pair_blocks = neighbour_check[
    neighbour_check["before_pair"].notna() &
    neighbour_check["after_pair"].notna() &
    (
        neighbour_check["before_pair"] ==
        neighbour_check["after_pair"]
    )
]

print(
    "Blocks with matching junction pair on both sides:",
    len(same_pair_blocks)
)

print(same_pair_blocks)

Blocks with matching junction pair on both sides: 43
    block  start_objectid  end_objectid  records  \
2      13           40488         40495        8   
3      71           44127         44134        8   
4      67           44105         44110        6   
7      64           44080         44084        5   
9      63           44073         44077        5   
10     11           40476         40479        4   
11     15           40502         40505        4   
12     66           44097         44100        4   
14     12           40482         40485        4   
15     69           44117         44120        4   
16     78           49181         49183        3   
19     48           42649         42651        3   
23     10           40470         40471        2   
25     58           44031         44032        2   
26     68           44113         44114        2   
29     62           44069         44070        2   
30     14           40498         40499        2   
31     72  

In [116]:
# ---------------------------------------------------------------
# Count the total number of missing records contained in blocks
# where both neighbouring records have the same junction pair.
# ---------------------------------------------------------------

print(
    "Recoverable blocks:",
    len(same_pair_blocks)
)

print(
    "Recoverable records:",
    same_pair_blocks["records"].sum()
)

print(
    "Remaining records after block recovery:",
    len(sec_remaining) -
    same_pair_blocks["records"].sum()
)

Recoverable blocks: 43
Recoverable records: 102
Remaining records after block recovery: 92


In [117]:
# ---------------------------------------------------------------
# Recover missing junctions for blocks where the record immediately
# before and immediately after the block contain the same
# fromjunction -> to junction pair.
#
# This is a structural recovery based on the ordering of the
# railway-network records, not a statistical guess.
# ---------------------------------------------------------------

for _, block in same_pair_blocks.iterrows():

    start_id = block["start_objectid"]
    end_id = block["end_objectid"]

    junction_pair = block["before_pair"]

    # Split the verified junction pair into its two endpoints.
    from_junction, to_junction = junction_pair.split(" → ", 1)

    # Update every missing record inside this contiguous block.
    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "South East Central Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Records recovered:", same_pair_blocks["records"].sum())

Records recovered: 102


In [118]:
# ---------------------------------------------------------------
# Verify the remaining missing SEC junction records.
# ---------------------------------------------------------------

sec_remaining_count = (
    (df["railwayzone"] == "South East Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print(
    "Remaining SEC missing-junction records:",
    sec_remaining_count
)

Remaining SEC missing-junction records: 92


In [119]:
# ---------------------------------------------------------------
# Isolate East Coast Railway records where both junctions
# are missing, then check which features are actually available.
# ---------------------------------------------------------------

ec_missing = df[
    (df["railwayzone"] == "East Coast Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

print("Missing-junction records:", len(ec_missing))

print("\nAvailable values:")
print(
    ec_missing[
        [
            "type",
            "speed",
            "noofstations",
            "tracklength",
            "nooflanes",
            "bridge_yn",
            "tunnel_yn",
            "st_lengthshape"
        ]
    ].notna().sum()
)

Missing-junction records: 105

Available values:
type              105
speed               0
noofstations       75
tracklength        75
nooflanes           0
bridge_yn         105
tunnel_yn         105
st_lengthshape    105
dtype: int64


In [120]:
# Quick categorical distributions.
for column in [
    "type",
    "nooflanes",
    "bridge_yn",
    "tunnel_yn"
]:
    print(f"\n--- {column} ---")
    print(ec_missing[column].value_counts(dropna=False))


--- type ---
type
Broad Gauge    105
Name: count, dtype: int64

--- nooflanes ---
nooflanes
NaN    105
Name: count, dtype: int64

--- bridge_yn ---
bridge_yn
N    58
Y    47
Name: count, dtype: int64

--- tunnel_yn ---
tunnel_yn
N    103
Y      2
Name: count, dtype: int64


In [121]:
# ---------------------------------------------------------------
# Inspect the variation of the available East Coast features.
# We need variation before using a feature for junction recovery.
# ---------------------------------------------------------------

print("\nNO. OF STATIONS:")
print(ec_missing["noofstations"].value_counts(dropna=False))

print("\nTRACK LENGTH:")
print(ec_missing["tracklength"].describe())

print("\nST LENGTH SHAPE:")
print(ec_missing["st_lengthshape"].describe())


NO. OF STATIONS:
noofstations
26.0    75
NaN     30
Name: count, dtype: int64

TRACK LENGTH:
count     75.0
mean     106.0
std        0.0
min      106.0
25%      106.0
50%      106.0
75%      106.0
max      106.0
Name: tracklength, dtype: float64

ST LENGTH SHAPE:
count      105.000000
mean       808.746146
std       2003.999793
min          5.849606
25%         26.835636
50%        137.863189
75%        537.501702
max      14103.302019
Name: st_lengthshape, dtype: float64


In [122]:
# ---------------------------------------------------------------
# Test whether bridge status + tunnel status + geometry-length
# range can uniquely identify a junction pair in East Coast Railway.
# ---------------------------------------------------------------

ec_complete = df[
    (df["railwayzone"] == "East Coast Railway") &
    df["fromjunction"].notna() &
    df["tojunction"].notna()
].copy()

# Create the junction-pair identifier.
ec_complete["junction_pair"] = (
    ec_complete["fromjunction"].astype(str)
    + " → "
    + ec_complete["tojunction"].astype(str)
)

# Use the same geometry-length bins.
bins = [
    0, 25, 50, 100, 250, 500,
    1000, 2500, 5000, 10000, float("inf")
]

labels = [
    "0-25", "25-50", "50-100", "100-250",
    "250-500", "500-1000", "1000-2500",
    "2500-5000", "5000-10000", "10000+"
]

ec_complete["length_bin"] = pd.cut(
    ec_complete["st_lengthshape"],
    bins=bins,
    labels=labels
)

ec_missing["length_bin"] = pd.cut(
    ec_missing["st_lengthshape"],
    bins=bins,
    labels=labels
)

match_features = [
    "bridge_yn",
    "tunnel_yn",
    "length_bin"
]

# Count how many different junction pairs exist for
# each feature combination.
pattern_counts = (
    ec_complete
    .dropna(subset=match_features)
    .groupby(match_features, observed=True)["junction_pair"]
    .nunique()
    .reset_index(name="unique_junction_pairs")
)

print(pattern_counts["unique_junction_pairs"].value_counts())

unique_junction_pairs
3     2
6     2
1     2
13    1
65    1
47    1
31    1
71    1
55    1
85    1
89    1
26    1
5     1
2     1
82    1
75    1
70    1
49    1
11    1
10    1
Name: count, dtype: int64


In [123]:
# ---------------------------------------------------------------
# Keep only feature combinations that map to exactly ONE
# known junction pair.
# ---------------------------------------------------------------

safe_patterns = pattern_counts[
    pattern_counts["unique_junction_pairs"] == 1
]

pattern_mapping = (
    ec_complete
    .dropna(
        subset=match_features + ["junction_pair"]
    )
    [
        match_features + ["junction_pair"]
    ]
    .drop_duplicates()
    .merge(
        safe_patterns[match_features],
        on=match_features,
        how="inner"
    )
)

ec_recovery_test = ec_missing.merge(
    pattern_mapping,
    on=match_features,
    how="left"
)

print(
    "Potentially recoverable:",
    ec_recovery_test["junction_pair"].notna().sum()
)

print(
    "No matching pattern:",
    ec_recovery_test["junction_pair"].isna().sum()
)

Potentially recoverable: 0
No matching pattern: 105


In [124]:
# ---------------------------------------------------------------
# Find contiguous blocks of East Coast records where both
# junctions are missing.
# ---------------------------------------------------------------

ec_remaining = df[
    (df["railwayzone"] == "East Coast Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

ec_remaining = ec_remaining.sort_values("objectid")

ec_remaining["objectid_diff"] = (
    ec_remaining["objectid"].diff()
)

ec_remaining["block"] = (
    ec_remaining["objectid_diff"] != 1
).cumsum()

ec_blocks = (
    ec_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(ec_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
7              117952        117990       39     18824.994960
8              117992        118023       32      8480.276576
2               13853         13873       21     16729.500960
6               22194         22197        4       333.297900
9              118025        118028        4      2493.820805
1               13851         13851        1      3442.028759
5               20888         20888        1      8344.715323
4               20885         20885        1     14103.302019
3               20883         20883        1      6493.254965
10             118297        118297        1      5673.153025


In [125]:
# ---------------------------------------------------------------
# Check whether each missing East Coast block is surrounded by
# the same known junction pair.
# ---------------------------------------------------------------

block_ranges = ec_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]

        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = (
                f"{b['fromjunction']} → {b['tojunction']}"
            )

    if len(after) > 0:
        a = after.iloc[0]

        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = (
                f"{a['fromjunction']} → {a['tojunction']}"
            )

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

ec_neighbour_check = pd.DataFrame(neighbour_results)

# Only retain blocks where both sides are known and identical.
ec_same_pair_blocks = ec_neighbour_check[
    ec_neighbour_check["before_pair"].notna() &
    ec_neighbour_check["after_pair"].notna() &
    (
        ec_neighbour_check["before_pair"] ==
        ec_neighbour_check["after_pair"]
    )
]

print(ec_same_pair_blocks)
print(
    "\nRecoverable records:",
    ec_same_pair_blocks["records"].sum()
)

   block  start_objectid  end_objectid  records  \
3      6           22194         22197        4   
4      9          118025        118028        4   

                       before_pair                       after_pair  
3  Duvvada → Samalkot Bypass Cabin  Duvvada → Samalkot Bypass Cabin  
4  Manoharabad Junction → Siddipet  Manoharabad Junction → Siddipet  

Recoverable records: 8


In [126]:
# ---------------------------------------------------------------
# Recover East Coast records where the missing block is surrounded
# by the same known junction pair on both sides.
# ---------------------------------------------------------------

for _, row in ec_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "East Coast Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Recovered:", ec_same_pair_blocks["records"].sum())

Recovered: 8


In [127]:
ec_remaining_count = (
    (df["railwayzone"] == "East Coast Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining East Coast missing junctions:", ec_remaining_count)

Remaining East Coast missing junctions: 97


In [128]:
# ---------------------------------------------------------------
# Isolate South Eastern Railway records where both junctions
# are missing and inspect which features are actually available.
# ---------------------------------------------------------------

ser_missing = df[
    (df["railwayzone"] == "South Eastern Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

print("Missing-junction records:", len(ser_missing))

print("\nAvailable values:")
print(
    ser_missing[
        [
            "type",
            "speed",
            "noofstations",
            "tracklength",
            "nooflanes",
            "bridge_yn",
            "tunnel_yn",
            "st_lengthshape"
        ]
    ].notna().sum()
)

for column in [
    "type",
    "nooflanes",
    "bridge_yn",
    "tunnel_yn"
]:
    print(f"\n--- {column} ---")
    print(ser_missing[column].value_counts(dropna=False))

Missing-junction records: 104

Available values:
type              104
speed             103
noofstations      103
tracklength       103
nooflanes         103
bridge_yn         104
tunnel_yn         104
st_lengthshape    104
dtype: int64

--- type ---
type
Broad Gauge    104
Name: count, dtype: int64

--- nooflanes ---
nooflanes
1.0    98
0.0     5
NaN     1
Name: count, dtype: int64

--- bridge_yn ---
bridge_yn
N    56
Y    48
Name: count, dtype: int64

--- tunnel_yn ---
tunnel_yn
N    104
Name: count, dtype: int64


In [129]:
# ---------------------------------------------------------------
# Check variation in the numerical/infrastructure features.
# Features with meaningful variation are candidates for junction
# recovery; constant features are discarded.
# ---------------------------------------------------------------

print("\nSPEED:")
print(ser_missing["speed"].value_counts(dropna=False))

print("\nNO. OF STATIONS:")
print(ser_missing["noofstations"].value_counts(dropna=False))

print("\nTRACK LENGTH:")
print(ser_missing["tracklength"].value_counts(dropna=False))

print("\nNO. OF LANES:")
print(ser_missing["nooflanes"].value_counts(dropna=False))

print("\nGEOMETRY LENGTH:")
print(ser_missing["st_lengthshape"].describe())


SPEED:
speed
0.0    103
NaN      1
Name: count, dtype: int64

NO. OF STATIONS:
noofstations
0.0    103
NaN      1
Name: count, dtype: int64

TRACK LENGTH:
tracklength
0.0    103
NaN      1
Name: count, dtype: int64

NO. OF LANES:
nooflanes
1.0    98
0.0     5
NaN     1
Name: count, dtype: int64

GEOMETRY LENGTH:
count      104.000000
mean       758.496485
std       1850.915487
min          5.926149
25%         15.154527
50%        162.806233
75%        689.548373
max      13928.470530
Name: st_lengthshape, dtype: float64


In [130]:
# ---------------------------------------------------------------
# Find contiguous blocks of South Eastern Railway records where
# both junctions are missing.
# ---------------------------------------------------------------

ser_remaining = df[
    (df["railwayzone"] == "South Eastern Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

ser_remaining = ser_remaining.sort_values("objectid")

ser_remaining["objectid_diff"] = (
    ser_remaining["objectid"].diff()
)

ser_remaining["block"] = (
    ser_remaining["objectid_diff"] != 1
).cumsum()

ser_blocks = (
    ser_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(ser_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
33             115102        115110        9      2239.780149
21             114995        115001        7       798.956989
11             114879        114884        6       811.650300
12             114887        114892        6      1618.668052
3               75394         75399        6      2045.325549
8               77166         77169        4      9153.119698
30             115067        115070        4      1024.042322
17             114972        114974        3       362.154396
9              114871        114873        3       689.990185
19             114989        114991        3      1188.185014
18             114981        114982        2       858.978749
26             115031        115032        2       253.038843
27             115037        115038        2       376.400118
29             115054        115055        2      1502.383119
4       

In [131]:
# ---------------------------------------------------------------
# Check whether each South Eastern Railway missing block is
# surrounded by the same known junction pair.
# ---------------------------------------------------------------

block_ranges = ser_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]
        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = (
                f"{b['fromjunction']} → {b['tojunction']}"
            )

    if len(after) > 0:
        a = after.iloc[0]
        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = (
                f"{a['fromjunction']} → {a['tojunction']}"
            )

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

ser_neighbour_check = pd.DataFrame(neighbour_results)

ser_same_pair_blocks = ser_neighbour_check[
    ser_neighbour_check["before_pair"].notna() &
    ser_neighbour_check["after_pair"].notna() &
    (
        ser_neighbour_check["before_pair"] ==
        ser_neighbour_check["after_pair"]
    )
]

print(
    "Recoverable blocks:",
    len(ser_same_pair_blocks)
)

print(
    "Recoverable records:",
    ser_same_pair_blocks["records"].sum()
)

print(ser_same_pair_blocks)

Recoverable blocks: 19
Recoverable records: 40
    block  start_objectid  end_objectid  records  \
0      33          115102        115110        9   
2      11          114879        114884        6   
6      30          115067        115070        4   
13     29          115054        115055        2   
15      5           75404         75405        2   
16      2           75388         75389        2   
20     34          115125        115126        2   
23     40          115171        115172        2   
25     16          114970        114970        1   
27     14          114940        114940        1   
29     10          114877        114877        1   
31      7           77122         77122        1   
34     23          115017        115017        1   
35     24          115024        115024        1   
36     20          114993        114993        1   
37     37          115146        115146        1   
41     41          115174        115174        1   
43     43        

In [132]:
# ---------------------------------------------------------------
# Recover South Eastern Railway missing-junction blocks where
# the same known junction pair appears immediately before and
# immediately after the missing block.
# ---------------------------------------------------------------

for _, row in ser_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "South Eastern Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Records recovered:", ser_same_pair_blocks["records"].sum())

Records recovered: 40


In [133]:
ser_remaining_count = (
    (df["railwayzone"] == "South Eastern Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining:", ser_remaining_count)

Remaining: 64


In [134]:
#East Central Railway

ecr_missing = df[
    (df["railwayzone"] == "East Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

print("Missing-junction records:", len(ecr_missing))

print("\nAvailable values:")
print(
    ecr_missing[
        [
            "type",
            "speed",
            "noofstations",
            "tracklength",
            "nooflanes",
            "bridge_yn",
            "tunnel_yn",
            "st_lengthshape"
        ]
    ].notna().sum()
)

for column in [
    "type",
    "nooflanes",
    "bridge_yn",
    "tunnel_yn"
]:
    print(f"\n--- {column} ---")
    print(ecr_missing[column].value_counts(dropna=False))

Missing-junction records: 79

Available values:
type              79
speed             78
noofstations      78
tracklength       78
nooflanes         79
bridge_yn         79
tunnel_yn         79
st_lengthshape    79
dtype: int64

--- type ---
type
Broad Gauge    68
               11
Name: count, dtype: int64

--- nooflanes ---
nooflanes
1.0    79
Name: count, dtype: int64

--- bridge_yn ---
bridge_yn
N    41
Y    38
Name: count, dtype: int64

--- tunnel_yn ---
tunnel_yn
N    79
Name: count, dtype: int64


In [135]:
print("\nSPEED:")
print(ecr_missing["speed"].value_counts(dropna=False))

print("\nNO. OF STATIONS:")
print(ecr_missing["noofstations"].value_counts(dropna=False))

print("\nTRACK LENGTH:")
print(ecr_missing["tracklength"].value_counts(dropna=False))

print("\nST LENGTH SHAPE:")
print(ecr_missing["st_lengthshape"].describe())


SPEED:
speed
0.0    78
NaN     1
Name: count, dtype: int64

NO. OF STATIONS:
noofstations
0.0    78
NaN     1
Name: count, dtype: int64

TRACK LENGTH:
tracklength
0.0    53
1.0    25
NaN     1
Name: count, dtype: int64

ST LENGTH SHAPE:
count       79.000000
mean      1009.693971
std       1852.724356
min         10.557071
25%         37.842101
50%        397.309577
75%       1064.019809
max      13155.353805
Name: st_lengthshape, dtype: float64


In [136]:
# ---------------------------------------------------------------
# Find contiguous blocks of East Central Railway records where
# both junctions are missing.
# ---------------------------------------------------------------

ecr_remaining = df[
    (df["railwayzone"] == "East Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

ecr_remaining = ecr_remaining.sort_values("objectid")

ecr_remaining["objectid_diff"] = (
    ecr_remaining["objectid"].diff()
)

ecr_remaining["block"] = (
    ecr_remaining["objectid_diff"] != 1
).cumsum()

ecr_blocks = (
    ecr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(ecr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
39              81132         81136        5      3136.916745
30              75893         75896        4      2544.972608
3               75492         75495        4      1285.267483
40              81141         81144        4      2989.579250
38              81128         81130        3       738.294606
2               75485         75487        3       352.932637
15              75729         75731        3      1966.943673
12              75708         75709        2       639.296614
10              75618         75619        2      1694.189835
23              75835         75836        2       973.123074
41              81160         81161        2      1672.736887
24              75845         75846        2       159.141166
33              75916         75917        2        67.483670
44              81192         81193        2       502.839020
43      

In [137]:
# ---------------------------------------------------------------
# Check whether an ECR missing block is surrounded by the same
# known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = ecr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]
        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = f"{b['fromjunction']} → {b['tojunction']}"

    if len(after) > 0:
        a = after.iloc[0]
        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = f"{a['fromjunction']} → {a['tojunction']}"

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

ecr_neighbour_check = pd.DataFrame(neighbour_results)

ecr_same_pair_blocks = ecr_neighbour_check[
    ecr_neighbour_check["before_pair"].notna() &
    ecr_neighbour_check["after_pair"].notna() &
    (
        ecr_neighbour_check["before_pair"] ==
        ecr_neighbour_check["after_pair"]
    )
]

print(
    "Recoverable blocks:",
    len(ecr_same_pair_blocks)
)

print(
    "Recoverable records:",
    ecr_same_pair_blocks["records"].sum()
)

print(ecr_same_pair_blocks)

Recoverable blocks: 16
Recoverable records: 28
    block  start_objectid  end_objectid  records  \
1      30           75893         75896        4   
3      40           81141         81144        4   
4      38           81128         81130        3   
8      10           75618         75619        2   
10     41           81160         81161        2   
11     24           75845         75846        2   
14     43           81175         81176        2   
32     25           75856         75856        1   
33     21           75777         75777        1   
35     35           75959         75959        1   
36     34           75919         75919        1   
39     28           75881         75881        1   
40     29           75891         75891        1   
41     26           75858         75858        1   
44     45           81209         81209        1   
45     46           81211         81211        1   

                            before_pair                           af

In [138]:
# ---------------------------------------------------------------
# Recover ECR missing-junction blocks where the same known
# junction pair appears immediately before and after the block.
# ---------------------------------------------------------------

for _, row in ecr_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "East Central Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Records recovered:", ecr_same_pair_blocks["records"].sum())

Records recovered: 28


In [139]:
ecr_remaining_count = (
    (df["railwayzone"] == "East Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining ECR missing junctions:", ecr_remaining_count)

Remaining ECR missing junctions: 51


In [140]:
# ---------------------------------------------------------------
# Isolate Eastern Railway records where both junctions are missing
# and inspect which attributes are actually usable.
# ---------------------------------------------------------------

er_missing = df[
    (df["railwayzone"] == "Eastern Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

print("Missing-junction records:", len(er_missing))

print("\nAvailable values:")
print(
    er_missing[
        [
            "type",
            "speed",
            "noofstations",
            "tracklength",
            "nooflanes",
            "bridge_yn",
            "tunnel_yn",
            "st_lengthshape"
        ]
    ].notna().sum()
)

for column in [
    "type",
    "nooflanes",
    "bridge_yn",
    "tunnel_yn"
]:
    print(f"\n--- {column} ---")
    print(er_missing[column].value_counts(dropna=False))

Missing-junction records: 78

Available values:
type              77
speed             55
noofstations      55
tracklength       55
nooflanes         75
bridge_yn         78
tunnel_yn         78
st_lengthshape    78
dtype: int64

--- type ---
type
Broad Gauge    77
NaN             1
Name: count, dtype: int64

--- nooflanes ---
nooflanes
1.0    72
NaN     3
0.0     3
Name: count, dtype: int64

--- bridge_yn ---
bridge_yn
N    41
Y    37
Name: count, dtype: int64

--- tunnel_yn ---
tunnel_yn
N    78
Name: count, dtype: int64


In [141]:
# ---------------------------------------------------------------
# Find contiguous blocks of Eastern Railway records where both
# junctions are missing.
# ---------------------------------------------------------------

er_remaining = df[
    (df["railwayzone"] == "Eastern Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

er_remaining = er_remaining.sort_values("objectid")

er_remaining["objectid_diff"] = (
    er_remaining["objectid"].diff()
)

er_remaining["block"] = (
    er_remaining["objectid_diff"] != 1
).cumsum()

er_blocks = (
    er_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(er_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
5               69124         69129        6      9311.924621
25              87070         87075        6      1945.278725
15              87004         87007        4      4202.682613
6               69132         69135        4       939.061086
21              87047         87050        4       748.128489
27              87081         87084        4      5805.899183
11              76820         76823        4     12177.897891
12              76826         76828        3     20456.718273
7               69139         69141        3       427.137455
13              77160         77162        3      1297.295191
16              87009         87010        2      1618.045055
20              87043         87044        2      3922.914329
17              87012         87013        2       941.846148
10              76814         76815        2     12928.905628
3       

In [142]:
# ---------------------------------------------------------------
# Check whether each Eastern Railway missing block is surrounded
# by the same known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = er_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]

        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = (
                f"{b['fromjunction']} → {b['tojunction']}"
            )

    if len(after) > 0:
        a = after.iloc[0]

        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = (
                f"{a['fromjunction']} → {a['tojunction']}"
            )

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

er_neighbour_check = pd.DataFrame(neighbour_results)

er_same_pair_blocks = er_neighbour_check[
    er_neighbour_check["before_pair"].notna() &
    er_neighbour_check["after_pair"].notna() &
    (
        er_neighbour_check["before_pair"] ==
        er_neighbour_check["after_pair"]
    )
]

print(
    "Recoverable blocks:",
    len(er_same_pair_blocks)
)

print(
    "Recoverable records:",
    er_same_pair_blocks["records"].sum()
)

print(er_same_pair_blocks)

Recoverable blocks: 19
Recoverable records: 49
    block  start_objectid  end_objectid  records  \
1      25           87070         87075        6   
2      15           87004         87007        4   
4      21           87047         87050        4   
5      27           87081         87084        4   
6      11           76820         76823        4   
7      12           76826         76828        3   
8       7           69139         69141        3   
10     16           87009         87010        2   
11     20           87043         87044        2   
12     17           87012         87013        2   
13     10           76814         76815        2   
15      4           69067         69068        2   
16     24           87060         87061        2   
19     28           87089         87090        2   
20     29           87095         87096        2   
24     30           87103         87104        2   
25     14           87002         87002        1   
27      8        

In [143]:
# ---------------------------------------------------------------
# Recover Eastern Railway records where the same known junction
# pair appears immediately before and after the missing block.
# ---------------------------------------------------------------

for _, row in er_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "Eastern Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Records recovered:", er_same_pair_blocks["records"].sum())

Records recovered: 49


In [144]:
er_remaining_count = (
    (df["railwayzone"] == "Eastern Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining:", er_remaining_count)

Remaining: 29


In [145]:
# ---------------------------------------------------------------
# Isolate North Western Railway records where both junctions
# are missing and inspect which features are actually available.
# ---------------------------------------------------------------

nwr_missing = df[
    (df["railwayzone"] == "North Western Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

print("Missing-junction records:", len(nwr_missing))

print("\nAvailable values:")
print(
    nwr_missing[
        [
            "type",
            "speed",
            "noofstations",
            "tracklength",
            "nooflanes",
            "bridge_yn",
            "tunnel_yn",
            "st_lengthshape"
        ]
    ].notna().sum()
)

for column in [
    "type",
    "nooflanes",
    "bridge_yn",
    "tunnel_yn"
]:
    print(f"\n--- {column} ---")
    print(nwr_missing[column].value_counts(dropna=False))

Missing-junction records: 76

Available values:
type              76
speed             76
noofstations      76
tracklength       76
nooflanes         76
bridge_yn         76
tunnel_yn          6
st_lengthshape    76
dtype: int64

--- type ---
type
Broad Gauge    76
Name: count, dtype: int64

--- nooflanes ---
nooflanes
0.0    73
1.0     3
Name: count, dtype: int64

--- bridge_yn ---
bridge_yn
Y    72
N     4
Name: count, dtype: int64

--- tunnel_yn ---
tunnel_yn
NaN    70
N       5
Y       1
Name: count, dtype: int64


In [146]:
# ---------------------------------------------------------------
# Find contiguous blocks of North Western Railway records where
# both junctions are missing.
# ---------------------------------------------------------------

nwr_remaining = df[
    (df["railwayzone"] == "North Western Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

nwr_remaining = nwr_remaining.sort_values("objectid")

nwr_remaining["objectid_diff"] = (
    nwr_remaining["objectid"].diff()
)

nwr_remaining["block"] = (
    nwr_remaining["objectid_diff"] != 1
).cumsum()

nwr_blocks = (
    nwr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(nwr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
18              89609         89665       57     26930.563123
13              80169         80170        2      3958.121916
15              89406         89407        2     12121.260024
1                6803          6803        1       272.810879
2                7053          7053        1        29.322614
3                7083          7083        1      1071.392885
7                7661          7661        1      1467.642381
4                7341          7341        1        37.576628
5                7369          7369        1       743.014973
6                7538          7538        1        62.281004
10               8368          8368        1        10.227889
9                8129          8129        1       874.750409
8                7841          7841        1        31.363871
11               8429          8429        1       333.651459
14      

In [147]:
# ---------------------------------------------------------------
# Check the records immediately before and after every NWR
# missing block.
#
# A block is considered recoverable only when both neighbouring
# records contain the same known fromjunction -> to junction pair.
# ---------------------------------------------------------------

block_ranges = nwr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]

        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = (
                f"{b['fromjunction']} → {b['tojunction']}"
            )

    if len(after) > 0:
        a = after.iloc[0]

        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = (
                f"{a['fromjunction']} → {a['tojunction']}"
            )

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

nwr_neighbour_check = pd.DataFrame(neighbour_results)

nwr_same_pair_blocks = nwr_neighbour_check[
    nwr_neighbour_check["before_pair"].notna() &
    nwr_neighbour_check["after_pair"].notna() &
    (
        nwr_neighbour_check["before_pair"] ==
        nwr_neighbour_check["after_pair"]
    )
]

print(
    "Recoverable blocks:",
    len(nwr_same_pair_blocks)
)

print(
    "Recoverable records:",
    nwr_same_pair_blocks["records"].sum()
)

print(nwr_same_pair_blocks)

Recoverable blocks: 13
Recoverable records: 13
    block  start_objectid  end_objectid  records  \
3       1            6803          6803        1   
4       2            7053          7053        1   
5       3            7083          7083        1   
6       7            7661          7661        1   
7       4            7341          7341        1   
9       6            7538          7538        1   
10     10            8368          8368        1   
11      9            8129          8129        1   
12      8            7841          7841        1   
13     11            8429          8429        1   
14     14           80185         80185        1   
15     12            8490          8490        1   
16     16           89547         89547        1   

                                          before_pair  \
3   Vanchi Maniyachchi Junction → Virudunagar Junc...   
4         Ponmalai (Golden Rock) → Thanjavur Junction   
5         Ponmalai (Golden Rock) → Thanjavur Junction

In [148]:
# ---------------------------------------------------------------
# Recover NWR blocks where the same known junction pair appears
# immediately before and after the missing record/block.
# ---------------------------------------------------------------

for _, row in nwr_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "North Western Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print(
    "Records recovered:",
    nwr_same_pair_blocks["records"].sum()
)

Records recovered: 13


In [149]:
nwr_remaining_count = (
    (df["railwayzone"] == "North Western Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining:", nwr_remaining_count)

Remaining: 63


In [150]:
# ---------------------------------------------------------------
# Find contiguous Central Railway records where both junctions
# are missing.
# ---------------------------------------------------------------

cr_remaining = df[
    (df["railwayzone"] == "Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

cr_remaining = cr_remaining.sort_values("objectid")

cr_remaining["objectid_diff"] = (
    cr_remaining["objectid"].diff()
)

cr_remaining["block"] = (
    cr_remaining["objectid_diff"] != 1
).cumsum()

cr_blocks = (
    cr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(cr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
4               38668         38676        9      8469.012982
5               38678         38683        6      1132.875460
22             111984        111989        6      1037.508343
10              48867         48870        4      1553.010980
21             111975        111978        4      1574.169305
17             111905        111907        3      3621.051721
19             111919        111920        2       760.457699
14             109990        109991        2       266.761420
13             109976        109977        2       188.193127
12             109934        109935        2      1388.701320
20             111956        111957        2       386.744709
3               38666         38666        1        44.445510
1               38659         38659        1       319.143089
2               38661         38661        1       120.675074
6       

In [151]:
# ---------------------------------------------------------------
# Check whether each Central Railway missing block is surrounded
# by the same known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = cr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]
        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = f"{b['fromjunction']} → {b['tojunction']}"

    if len(after) > 0:
        a = after.iloc[0]
        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = f"{a['fromjunction']} → {a['tojunction']}"

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

cr_neighbour_check = pd.DataFrame(neighbour_results)

cr_same_pair_blocks = cr_neighbour_check[
    cr_neighbour_check["before_pair"].notna() &
    cr_neighbour_check["after_pair"].notna() &
    (
        cr_neighbour_check["before_pair"] ==
        cr_neighbour_check["after_pair"]
    )
]

print(
    "Recoverable blocks:",
    len(cr_same_pair_blocks)
)

print(
    "Recoverable records:",
    cr_same_pair_blocks["records"].sum()
)

print(cr_same_pair_blocks)

Recoverable blocks: 7
Recoverable records: 21
    block  start_objectid  end_objectid  records  \
0       4           38668         38676        9   
1       5           38678         38683        6   
8      13          109976        109977        2   
11      3           38666         38666        1   
15      7           38689         38689        1   
18     11          109338        109338        1   
19     18          111916        111916        1   

                                      before_pair  \
0          Chhindwara Junction → Nainpur Junction   
1          Chhindwara Junction → Nainpur Junction   
8                          Panvel Junction → Roha   
11         Chhindwara Junction → Nainpur Junction   
15         Chhindwara Junction → Nainpur Junction   
18  Panvel Junction → Jawaharlal Nehru Port Trust   
19                              Uran City → Nerul   

                                       after_pair  
0          Chhindwara Junction → Nainpur Junction  
1       

In [152]:
# ---------------------------------------------------------------
# Recover Central Railway records where the same known junction
# pair appears immediately before and after the missing block.
# ---------------------------------------------------------------

for _, row in cr_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "Central Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print(
    "Records recovered:",
    cr_same_pair_blocks["records"].sum()
)

Records recovered: 21


In [153]:
cr_remaining_count = (
    (df["railwayzone"] == "Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining:", cr_remaining_count)

Remaining: 32


In [154]:
# ---------------------------------------------------------------
# Find contiguous West Central Railway records where both
# junctions are missing.
# ---------------------------------------------------------------

wcr_remaining = df[
    (df["railwayzone"] == "West Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

wcr_remaining = wcr_remaining.sort_values("objectid")

wcr_remaining["objectid_diff"] = (
    wcr_remaining["objectid"].diff()
)

wcr_remaining["block"] = (
    wcr_remaining["objectid_diff"] != 1
).cumsum()

wcr_blocks = (
    wcr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(wcr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
17              48696         48701        6      5134.342742
12              41747         41752        6      4495.846800
13              47205         47209        5      1817.141013
8               41715         41717        3      1804.039158
18              82566         82567        2      1112.414845
9               41724         41725        2      1055.192255
4               40344         40345        2       641.529780
10              41730         41731        2       339.353935
3               40321         40321        1       803.907481
1               40070         40070        1      3641.331773
2               40284         40284        1       588.078225
11              41740         41740        1      1300.773936
5               40429         40429        1      3364.486006
6               41706         41706        1        13.493368
7       

In [155]:
# ---------------------------------------------------------------
# Check whether each West Central Railway missing block is
# surrounded by the same known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = wcr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]
        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = f"{b['fromjunction']} → {b['tojunction']}"

    if len(after) > 0:
        a = after.iloc[0]
        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = f"{a['fromjunction']} → {a['tojunction']}"

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

wcr_neighbour_check = pd.DataFrame(neighbour_results)

wcr_same_pair_blocks = wcr_neighbour_check[
    wcr_neighbour_check["before_pair"].notna() &
    wcr_neighbour_check["after_pair"].notna() &
    (
        wcr_neighbour_check["before_pair"] ==
        wcr_neighbour_check["after_pair"]
    )
]

print(
    "Recoverable blocks:",
    len(wcr_same_pair_blocks)
)

print(
    "Recoverable records:",
    wcr_same_pair_blocks["records"].sum()
)

print(wcr_same_pair_blocks)

Recoverable blocks: 5
Recoverable records: 9
    block  start_objectid  end_objectid  records  \
3       8           41715         41717        3   
4      18           82566         82567        2   
6       4           40344         40345        2   
15     15           48692         48692        1   
18     19           82576         82576        1   

                                before_pair  \
3                    Jabalpur → Katni South   
4   Gurla N Cabin → Sawai Madhopur Junction   
6                     Lidhora Khurd → Rithi   
15                   Sagma → Satna Junction   
18  Gurla N Cabin → Sawai Madhopur Junction   

                                 after_pair  
3                    Jabalpur → Katni South  
4   Gurla N Cabin → Sawai Madhopur Junction  
6                     Lidhora Khurd → Rithi  
15                   Sagma → Satna Junction  
18  Gurla N Cabin → Sawai Madhopur Junction  


In [156]:
# ---------------------------------------------------------------
# Recover WCR records where the same known junction pair appears
# immediately before and after the missing block.
# ---------------------------------------------------------------

for _, row in wcr_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "West Central Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Records recovered:", wcr_same_pair_blocks["records"].sum())

Records recovered: 9


In [157]:
wcr_remaining_count = (
    (df["railwayzone"] == "West Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining:", wcr_remaining_count)

Remaining: 30


In [158]:
# ---------------------------------------------------------------
# Find contiguous South Central Railway records where both
# junctions are missing.
# ---------------------------------------------------------------

scr_remaining = df[
    (df["railwayzone"] == "South Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

scr_remaining = scr_remaining.sort_values("objectid")

scr_remaining["objectid_diff"] = (
    scr_remaining["objectid"].diff()
)

scr_remaining["block"] = (
    scr_remaining["objectid_diff"] != 1
).cumsum()

scr_blocks = (
    scr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(scr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
3               94897         94901        5     17888.338642
12             111648        111651        4      1720.273186
4               97308         97310        3      6288.384870
2               77163         77165        3      9417.473094
10             111624        111625        2       575.102882
5               97455         97456        2      1580.756509
11             111637        111638        2      2722.962676
14             117836        117837        2     15324.686437
6              101100        101100        1     13888.611499
1               17591         17591        1      2912.199744
9              111617        111617        1       797.395743
8              111508        111508        1         9.744174
7              101102        101102        1     11942.993674
13             111658        111658        1      2081.885462


In [159]:
# ---------------------------------------------------------------
# Check whether each South Central Railway missing block is
# surrounded by the same known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = scr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]

        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = (
                f"{b['fromjunction']} → {b['tojunction']}"
            )

    if len(after) > 0:
        a = after.iloc[0]

        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = (
                f"{a['fromjunction']} → {a['tojunction']}"
            )

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

scr_neighbour_check = pd.DataFrame(neighbour_results)

scr_same_pair_blocks = scr_neighbour_check[
    scr_neighbour_check["before_pair"].notna() &
    scr_neighbour_check["after_pair"].notna() &
    (
        scr_neighbour_check["before_pair"] ==
        scr_neighbour_check["after_pair"]
    )
]

print(
    "Recoverable blocks:",
    len(scr_same_pair_blocks)
)

print(
    "Recoverable records:",
    scr_same_pair_blocks["records"].sum()
)

print(scr_same_pair_blocks)

Recoverable blocks: 5
Recoverable records: 10
    block  start_objectid  end_objectid  records  \
1      12          111648        111651        4   
4      10          111624        111625        2   
6      11          111637        111638        2   
10      9          111617        111617        1   
13     13          111658        111658        1   

                             before_pair                            after_pair  
1     Parli Vaijnath → Parbhani Junction    Parli Vaijnath → Parbhani Junction  
4   Parli Vaijnath → Latur Road Junction  Parli Vaijnath → Latur Road Junction  
6     Parli Vaijnath → Parbhani Junction    Parli Vaijnath → Parbhani Junction  
10  Parli Vaijnath → Latur Road Junction  Parli Vaijnath → Latur Road Junction  
13    Parli Vaijnath → Parbhani Junction    Parli Vaijnath → Parbhani Junction  


In [160]:
# ---------------------------------------------------------------
# Recover SCR records where the same known junction pair appears
# immediately before and after the missing block.
# ---------------------------------------------------------------

for _, row in scr_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "South Central Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Records recovered:", scr_same_pair_blocks["records"].sum())

Records recovered: 10


In [161]:
scr_remaining_count = (
    (df["railwayzone"] == "South Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining:", scr_remaining_count)

Remaining: 19


In [162]:
# ---------------------------------------------------------------
# Find contiguous Western Railway records where both junctions
# are missing.
# ---------------------------------------------------------------

wr_remaining = df[
    (df["railwayzone"] == "Western Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

wr_remaining = wr_remaining.sort_values("objectid")

wr_remaining["objectid_diff"] = (
    wr_remaining["objectid"].diff()
)

wr_remaining["block"] = (
    wr_remaining["objectid_diff"] != 1
).cumsum()

wr_blocks = (
    wr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(wr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
9               97257         97269       13     13352.410671
3               43426         43427        2      1100.210755
6               43436         43437        2      2837.879672
1               43418         43418        1       226.371395
4               43429         43429        1      1269.581306
2               43420         43420        1        37.053602
7               43444         43444        1      1630.396553
5               43431         43431        1        63.906258
8               80613         80613        1      1422.452317
10             102424        102424        1      4575.896948
11             117925        117925        1      2537.139432
12             117928        117928        1     14179.161576


In [163]:
# ---------------------------------------------------------------
# Check whether each Western Railway missing block is surrounded
# by the same known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = wr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]

        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = (
                f"{b['fromjunction']} → {b['tojunction']}"
            )

    if len(after) > 0:
        a = after.iloc[0]

        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = (
                f"{a['fromjunction']} → {a['tojunction']}"
            )

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

wr_neighbour_check = pd.DataFrame(neighbour_results)

wr_same_pair_blocks = wr_neighbour_check[
    wr_neighbour_check["before_pair"].notna() &
    wr_neighbour_check["after_pair"].notna() &
    (
        wr_neighbour_check["before_pair"] ==
        wr_neighbour_check["after_pair"]
    )
]

print(
    "Recoverable blocks:",
    len(wr_same_pair_blocks)
)

print(
    "Recoverable records:",
    wr_same_pair_blocks["records"].sum()
)

print(wr_same_pair_blocks)

Recoverable blocks: 7
Recoverable records: 9
   block  start_objectid  end_objectid  records  \
1      3           43426         43427        2   
2      6           43436         43437        2   
3      1           43418         43418        1   
4      4           43429         43429        1   
5      2           43420         43420        1   
6      7           43444         43444        1   
7      5           43431         43431        1   

                      before_pair                      after_pair  
1  Nimach → Chittaurgarh Junction  Nimach → Chittaurgarh Junction  
2  Nimach → Chittaurgarh Junction  Nimach → Chittaurgarh Junction  
3  Nimach → Chittaurgarh Junction  Nimach → Chittaurgarh Junction  
4  Nimach → Chittaurgarh Junction  Nimach → Chittaurgarh Junction  
5  Nimach → Chittaurgarh Junction  Nimach → Chittaurgarh Junction  
6  Nimach → Chittaurgarh Junction  Nimach → Chittaurgarh Junction  
7  Nimach → Chittaurgarh Junction  Nimach → Chittaurgarh Junction  


In [164]:
# ---------------------------------------------------------------
# Recover Western Railway records where the same known junction
# pair appears immediately before and after the missing block.
# ---------------------------------------------------------------

for _, row in wr_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "Western Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Records recovered:", wr_same_pair_blocks["records"].sum())

Records recovered: 9


In [165]:
wr_remaining_count = (
    (df["railwayzone"] == "Western Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining:", wr_remaining_count)

Remaining: 17


In [166]:
# ---------------------------------------------------------------
# Find contiguous North Central Railway records where both
# junctions are missing.
# ---------------------------------------------------------------

ncr_remaining = df[
    (df["railwayzone"] == "North Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

ncr_remaining = ncr_remaining.sort_values("objectid")

ncr_remaining["objectid_diff"] = (
    ncr_remaining["objectid"].diff()
)

ncr_remaining["block"] = (
    ncr_remaining["objectid_diff"] != 1
).cumsum()

ncr_blocks = (
    ncr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(ncr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
3               48576         48579        4      1541.067888
12              48644         48646        3      4313.965910
7               48604         48606        3      1527.085139
5               48596         48597        2       565.693411
4               48586         48587        2      2881.179936
11              48641         48642        2       592.380142
6               48600         48601        2       474.182843
10              48631         48632        2       381.102352
1               33982         33982        1      3122.064141
2               45867         45867        1         7.823330
8               48624         48624        1       682.602677
9               48626         48626        1        16.246131
13              48652         48652        1       598.704244


In [167]:
# ---------------------------------------------------------------
# Check whether each North Central Railway missing block is
# surrounded by the same known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = ncr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]

        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = (
                f"{b['fromjunction']} → {b['tojunction']}"
            )

    if len(after) > 0:
        a = after.iloc[0]

        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = (
                f"{a['fromjunction']} → {a['tojunction']}"
            )

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

ncr_neighbour_check = pd.DataFrame(neighbour_results)

ncr_same_pair_blocks = ncr_neighbour_check[
    ncr_neighbour_check["before_pair"].notna() &
    ncr_neighbour_check["after_pair"].notna() &
    (
        ncr_neighbour_check["before_pair"] ==
        ncr_neighbour_check["after_pair"]
    )
]

print("Recoverable blocks:", len(ncr_same_pair_blocks))
print("Recoverable records:", ncr_same_pair_blocks["records"].sum())
print(ncr_same_pair_blocks)

Recoverable blocks: 7
Recoverable records: 16
    block  start_objectid  end_objectid  records  \
0       3           48576         48579        4   
1      12           48644         48646        3   
2       7           48604         48606        3   
4       4           48586         48587        2   
6       6           48600         48601        2   
8       1           33982         33982        1   
10      8           48624         48624        1   

                                          before_pair  \
0                    Guna Junction → Gwalior Junction   
1                    Guna Junction → Gwalior Junction   
2                            Gwalior Junction → Antri   
4                    Guna Junction → Gwalior Junction   
6                            Gwalior Junction → Antri   
8   Ait Junction → Virangana Lakshmibai Jhansi Jun...   
10                           Gwalior Junction → Antri   

                                           after_pair  
0                    Gun

In [168]:
# ---------------------------------------------------------------
# Recover NCR records where the same known junction pair appears
# immediately before and after the missing block.
# ---------------------------------------------------------------

for _, row in ncr_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "North Central Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Records recovered:", ncr_same_pair_blocks["records"].sum())

Records recovered: 16


In [169]:
ncr_remaining_count = (
    (df["railwayzone"] == "North Central Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining:", ncr_remaining_count)

Remaining: 9


In [170]:
# ---------------------------------------------------------------
# Find contiguous South Western Railway records where both
# junctions are missing.
# ---------------------------------------------------------------

swr_remaining = df[
    (df["railwayzone"] == "South Western Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

swr_remaining = swr_remaining.sort_values("objectid")

swr_remaining["objectid_diff"] = (
    swr_remaining["objectid"].diff()
)

swr_remaining["block"] = (
    swr_remaining["objectid_diff"] != 1
).cumsum()

swr_blocks = (
    swr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(swr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
1                3061          3077       17      8229.723768
3               19015         19016        2      8906.359203
2                3079          3079        1      1599.565625
4               19021         19021        1       513.721778


In [171]:
# ---------------------------------------------------------------
# Check whether each South Western Railway missing block is
# surrounded by the same known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = swr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]
        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = f"{b['fromjunction']} → {b['tojunction']}"

    if len(after) > 0:
        a = after.iloc[0]
        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = f"{a['fromjunction']} → {a['tojunction']}"

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

swr_neighbour_check = pd.DataFrame(neighbour_results)

swr_same_pair_blocks = swr_neighbour_check[
    swr_neighbour_check["before_pair"].notna() &
    swr_neighbour_check["after_pair"].notna() &
    (
        swr_neighbour_check["before_pair"] ==
        swr_neighbour_check["after_pair"]
    )
]

print("Recoverable blocks:", len(swr_same_pair_blocks))
print("Recoverable records:", swr_same_pair_blocks["records"].sum())
print(swr_same_pair_blocks)

Recoverable blocks: 4
Recoverable records: 21
   block  start_objectid  end_objectid  records  \
0      1            3061          3077       17   
1      3           19015         19016        2   
2      2            3079          3079        1   
3      4           19021         19021        1   

                                      before_pair  \
0  Bangarapet Junction → Jolarpettai Bypass Cabin   
1  Bangarapet Junction → Jolarpettai Bypass Cabin   
2  Bangarapet Junction → Jolarpettai Bypass Cabin   
3  Bangarapet Junction → Jolarpettai Bypass Cabin   

                                       after_pair  
0  Bangarapet Junction → Jolarpettai Bypass Cabin  
1  Bangarapet Junction → Jolarpettai Bypass Cabin  
2  Bangarapet Junction → Jolarpettai Bypass Cabin  
3  Bangarapet Junction → Jolarpettai Bypass Cabin  


In [172]:
# ---------------------------------------------------------------
# Recover all SWR missing junctions using the matching junction
# pair found immediately before and after each missing block.
# ---------------------------------------------------------------

for _, row in swr_same_pair_blocks.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    from_junction, to_junction = row["before_pair"].split(
        " → ", 1
    )

    mask = (
        (df["objectid"] >= start_id) &
        (df["objectid"] <= end_id) &
        (df["railwayzone"] == "South Western Railway") &
        df["fromjunction"].isna() &
        df["tojunction"].isna()
    )

    df.loc[mask, "fromjunction"] = from_junction
    df.loc[mask, "tojunction"] = to_junction

print("Records recovered:", swr_same_pair_blocks["records"].sum())

Records recovered: 21


In [173]:
swr_remaining_count = (
    (df["railwayzone"] == "South Western Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
).sum()

print("Remaining:", swr_remaining_count)

Remaining: 0


In [174]:
# ---------------------------------------------------------------
# Find contiguous Southern Railway records where both junctions
# are missing.
# ---------------------------------------------------------------

sr_remaining = df[
    (df["railwayzone"] == "Southern Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

sr_remaining = sr_remaining.sort_values("objectid")

sr_remaining["objectid_diff"] = (
    sr_remaining["objectid"].diff()
)

sr_remaining["block"] = (
    sr_remaining["objectid_diff"] != 1
).cumsum()

sr_blocks = (
    sr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(sr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
4                3817          3820        4      8353.801721
6                6793          6795        3      6343.692187
1                 814           814        1      6207.565998
3                3782          3782        1      2582.303656
2                3778          3778        1       846.857060
5                3822          3822        1        21.710341
7                6797          6797        1      6363.875437


In [175]:
# ---------------------------------------------------------------
# Check whether each Southern Railway missing block is surrounded
# by the same known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = sr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]

        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = f"{b['fromjunction']} → {b['tojunction']}"

    if len(after) > 0:
        a = after.iloc[0]

        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = f"{a['fromjunction']} → {a['tojunction']}"

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

sr_neighbour_check = pd.DataFrame(neighbour_results)

sr_same_pair_blocks = sr_neighbour_check[
    sr_neighbour_check["before_pair"].notna() &
    sr_neighbour_check["after_pair"].notna() &
    (
        sr_neighbour_check["before_pair"] ==
        sr_neighbour_check["after_pair"]
    )
]

print("Recoverable blocks:", len(sr_same_pair_blocks))
print("Recoverable records:", sr_same_pair_blocks["records"].sum())
print(sr_same_pair_blocks)

Recoverable blocks: 0
Recoverable records: 0
Empty DataFrame
Columns: [block, start_objectid, end_objectid, records, before_pair, after_pair]
Index: []


In [176]:
# ---------------------------------------------------------------
# Find contiguous Northern Railway records where both junctions
# are missing.
# ---------------------------------------------------------------

nr_remaining = df[
    (df["railwayzone"] == "Northern Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

nr_remaining = nr_remaining.sort_values("objectid")

nr_remaining["objectid_diff"] = nr_remaining["objectid"].diff()

nr_remaining["block"] = (
    nr_remaining["objectid_diff"] != 1
).cumsum()

nr_blocks = (
    nr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(nr_blocks)

       start_objectid  end_objectid  records  geometry_length
block                                                        
3               34480         34481        2      9026.740218
1               30460         30460        1      9818.172157
2               33977         33977        1      2716.635468
4               88948         88948        1      2197.458848


In [177]:
# ---------------------------------------------------------------
# Check whether each Northern Railway missing block is surrounded
# by the same known junction pair on both sides.
# ---------------------------------------------------------------

block_ranges = nr_blocks.reset_index()[
    ["block", "start_objectid", "end_objectid", "records"]
]

neighbour_results = []

for _, row in block_ranges.iterrows():

    start_id = row["start_objectid"]
    end_id = row["end_objectid"]

    before = df[df["objectid"] == start_id - 1]
    after = df[df["objectid"] == end_id + 1]

    before_pair = None
    after_pair = None

    if len(before) > 0:
        b = before.iloc[0]

        if pd.notna(b["fromjunction"]) and pd.notna(b["tojunction"]):
            before_pair = f"{b['fromjunction']} → {b['tojunction']}"

    if len(after) > 0:
        a = after.iloc[0]

        if pd.notna(a["fromjunction"]) and pd.notna(a["tojunction"]):
            after_pair = f"{a['fromjunction']} → {a['tojunction']}"

    neighbour_results.append({
        "block": row["block"],
        "start_objectid": start_id,
        "end_objectid": end_id,
        "records": row["records"],
        "before_pair": before_pair,
        "after_pair": after_pair
    })

nr_neighbour_check = pd.DataFrame(neighbour_results)

nr_same_pair_blocks = nr_neighbour_check[
    nr_neighbour_check["before_pair"].notna() &
    nr_neighbour_check["after_pair"].notna() &
    (
        nr_neighbour_check["before_pair"] ==
        nr_neighbour_check["after_pair"]
    )
]

print("Recoverable blocks:", len(nr_same_pair_blocks))
print("Recoverable records:", nr_same_pair_blocks["records"].sum())
print(nr_same_pair_blocks)

Recoverable blocks: 0
Recoverable records: 0
Empty DataFrame
Columns: [block, start_objectid, end_objectid, records, before_pair, after_pair]
Index: []


In [178]:
df['railwayzone'].unique()

<StringArray>
[          'Southern Railway',           'Northern Railway',
            'Western Railway',      'South Western Railway',
 'South East Central Railway',      'South Central Railway',
      'North Central Railway',      'South Eastern Railway',
            'Eastern Railway',      'North Western Railway',
         'East Coast Railway',            'Central Railway',
             'Konkan Railway',                         <NA>,
      'North Eastern Railway',       'East Central Railway',
       'West Central Railway', 'Northeast Frontier Railway']
Length: 18, dtype: string

In [179]:
# ---------------------------------------------------------------
# Find contiguous Konkan Railway records where both junctions
# are missing.
# ---------------------------------------------------------------

konkan_remaining = df[
    (df["railwayzone"] == "Konkan Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

konkan_remaining = konkan_remaining.sort_values("objectid")

konkan_remaining["objectid_diff"] = (
    konkan_remaining["objectid"].diff()
)

konkan_remaining["block"] = (
    konkan_remaining["objectid_diff"] != 1
).cumsum()

konkan_blocks = (
    konkan_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(konkan_blocks)

Empty DataFrame
Columns: [start_objectid, end_objectid, records, geometry_length]
Index: []


In [180]:
# ---------------------------------------------------------------
# Find contiguous North Eastern Railway records where both
# junctions are missing.
# ---------------------------------------------------------------

ner_remaining = df[
    (df["railwayzone"] == "North Eastern Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

ner_remaining = ner_remaining.sort_values("objectid")

ner_remaining["objectid_diff"] = (
    ner_remaining["objectid"].diff()
)

ner_remaining["block"] = (
    ner_remaining["objectid_diff"] != 1
).cumsum()

ner_blocks = (
    ner_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(ner_blocks)

Empty DataFrame
Columns: [start_objectid, end_objectid, records, geometry_length]
Index: []


In [181]:
# ---------------------------------------------------------------
# Find contiguous Northeast Frontier Railway records where both
# junctions are missing.
# ---------------------------------------------------------------

nfr_remaining = df[
    (df["railwayzone"] == "Northeast Frontier Railway") &
    df["fromjunction"].isna() &
    df["tojunction"].isna()
].copy()

nfr_remaining = nfr_remaining.sort_values("objectid")

nfr_remaining["objectid_diff"] = (
    nfr_remaining["objectid"].diff()
)

nfr_remaining["block"] = (
    nfr_remaining["objectid_diff"] != 1
).cumsum()

nfr_blocks = (
    nfr_remaining
    .groupby("block")
    .agg(
        start_objectid=("objectid", "min"),
        end_objectid=("objectid", "max"),
        records=("objectid", "count"),
        geometry_length=("st_lengthshape", "sum")
    )
    .sort_values("records", ascending=False)
)

print(nfr_blocks)

Empty DataFrame
Columns: [start_objectid, end_objectid, records, geometry_length]
Index: []


<h2>Status Check</h2>

In [182]:
# ===============================================================
# FINAL DATASET PROGRESS CHECK
# ===============================================================
# This cell only reports the current state of the dataframe.
# It does NOT modify df.
# ===============================================================

print("=" * 60)
print("DATASET PROGRESS CHECK")
print("=" * 60)

# ---------------------------------------------------------------
# 1. Overall dataset size
# ---------------------------------------------------------------

print("\n--- DATASET SIZE ---")
print("Total records :", len(df))
print("Total columns :", len(df.columns))


# ---------------------------------------------------------------
# 2. Railway zones currently present
# ---------------------------------------------------------------

print("\n--- RAILWAY ZONES ---")

zone_counts = df["railwayzone"].value_counts(dropna=False)

print(zone_counts)

print("\nNumber of valid railway zones:",
      df["railwayzone"].dropna().nunique())

print("Missing railwayzone:",
      df["railwayzone"].isna().sum())


# ---------------------------------------------------------------
# 3. Junction completeness
# ---------------------------------------------------------------

print("\n--- JUNCTION COMPLETENESS ---")

both_present = (
    df["fromjunction"].notna() &
    df["tojunction"].notna()
)

both_missing = (
    df["fromjunction"].isna() &
    df["tojunction"].isna()
)

from_missing = df["fromjunction"].isna()
to_missing = df["tojunction"].isna()

print("Both junctions present :", both_present.sum())
print("Both junctions missing :", both_missing.sum())
print("From junction missing  :", from_missing.sum())
print("To junction missing    :", to_missing.sum())


# ---------------------------------------------------------------
# 4. Junction completeness percentage
# ---------------------------------------------------------------

print("\n--- JUNCTION COMPLETENESS % ---")

print(
    "Complete junction records:",
    round(both_present.mean() * 100, 2), "%"
)

print(
    "Incomplete junction records:",
    round((1 - both_present.mean()) * 100, 2), "%"
)


# ---------------------------------------------------------------
# 5. Remaining missing junctions by railway zone
# ---------------------------------------------------------------

print("\n--- REMAINING MISSING JUNCTIONS BY ZONE ---")

remaining_by_zone = (
    df[both_missing]
    .groupby("railwayzone", dropna=False)
    .size()
    .sort_values(ascending=False)
)

print(remaining_by_zone)


# ---------------------------------------------------------------
# 6. How many records were removed because all key information
#    was missing?
# ---------------------------------------------------------------

print("\n--- CURRENT NULL PATTERN ---")

print(
    df[
        ["fromjunction", "tojunction", "railwayzone"]
    ].isna().value_counts()
)


# ---------------------------------------------------------------
# 7. Overall missing-value summary
# ---------------------------------------------------------------

print("\n--- MISSING VALUES BY COLUMN ---")

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (
        df.isna().mean() * 100
    ).round(2)
})

print(
    missing_summary.sort_values(
        "missing_count",
        ascending=False
    )
)


# ---------------------------------------------------------------
# 8. Number of unique junctions
# ---------------------------------------------------------------

print("\n--- JUNCTION COUNTS ---")

print(
    "Unique from-junctions:",
    df["fromjunction"].nunique()
)

print(
    "Unique to-junctions:",
    df["tojunction"].nunique()
)

# Unique junctions considering both sides together
all_junctions = pd.concat([
    df["fromjunction"],
    df["tojunction"]
]).dropna().unique()

print(
    "Unique junctions overall:",
    len(all_junctions)
)


# ---------------------------------------------------------------
# 9. Unique junction pairs
# ---------------------------------------------------------------

print("\n--- JUNCTION PAIRS ---")

complete_pairs = df.loc[
    both_present,
    ["fromjunction", "tojunction"]
].drop_duplicates()

print(
    "Unique complete junction pairs:",
    len(complete_pairs)
)


# ---------------------------------------------------------------
# 10. Duplicate records
# ---------------------------------------------------------------

print("\n--- DUPLICATES ---")

print(
    "Completely duplicated rows:",
    df.duplicated().sum()
)

print(
    "Duplicated objectids:",
    df["objectid"].duplicated().sum()
)

print(
    "Duplicated IDs:",
    df["id"].duplicated().sum()
)


# ---------------------------------------------------------------
# 11. ObjectID continuity
# ---------------------------------------------------------------

print("\n--- OBJECTID ---")

print("Minimum objectid:", df["objectid"].min())
print("Maximum objectid:", df["objectid"].max())
print("Unique objectids:", df["objectid"].nunique())


# ---------------------------------------------------------------
# 12. Final recovery summary
# ---------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print("Total records:", len(df))
print("Complete junction records:", both_present.sum())
print("Remaining incomplete junction records:", both_missing.sum())

print(
    "Junction completeness:",
    round(both_present.mean() * 100, 2),
    "%"
)

print(
    "Valid railway zones:",
    df["railwayzone"].dropna().nunique()
)

print("=" * 60)

DATASET PROGRESS CHECK

--- DATASET SIZE ---
Total records : 119340
Total columns : 14

--- RAILWAY ZONES ---
railwayzone
South Central Railway         15090
Western Railway               14210
South Western Railway          8475
Southern Railway               8016
North Western Railway          7768
Northern Railway               6862
West Central Railway           6830
East Coast Railway             6636
East Central Railway           6386
Northeast Frontier Railway     6232
Central Railway                6090
North Central Railway          6035
South Eastern Railway          4921
South East Central Railway     4525
North Eastern Railway          3715
Eastern Railway                3521
<NA>                           2602
Konkan Railway                 1426
Name: count, dtype: Int64

Number of valid railway zones: 17
Missing railwayzone: 2602

--- JUNCTION COMPLETENESS ---
Both junctions present : 118803
Both junctions missing : 537
From junction missing  : 537
To junction missing   

In [183]:
# ===============================================================
# CREATE GRAPH-READY DATASET
# ===============================================================

graph_df = df.copy()

print("Initial records:", len(graph_df))

# ---------------------------------------------------------------
# 1. Remove records with no railway zone
# ---------------------------------------------------------------

graph_df = graph_df[
    graph_df["railwayzone"].notna()
].copy()

print("After removing missing railwayzone:", len(graph_df))


# ---------------------------------------------------------------
# 2. Remove records where either junction is missing
# ---------------------------------------------------------------

graph_df = graph_df[
    graph_df["fromjunction"].notna() &
    graph_df["tojunction"].notna()
].copy()

print("After removing incomplete junctions:", len(graph_df))


# ---------------------------------------------------------------
# 3. Remove invalid/zero-length geometry
# ---------------------------------------------------------------

graph_df = graph_df[
    graph_df["st_lengthshape"].notna() &
    (graph_df["st_lengthshape"] > 0)
].copy()

print("After removing invalid geometry:", len(graph_df))


# ---------------------------------------------------------------
# 4. Remove exact duplicate records
# ---------------------------------------------------------------

graph_df = graph_df.drop_duplicates().copy()

print("After removing exact duplicates:", len(graph_df))


# ---------------------------------------------------------------
# 5. Reset index
# ---------------------------------------------------------------

graph_df = graph_df.reset_index(drop=True)


# ---------------------------------------------------------------
# FINAL CHECK
# ---------------------------------------------------------------

print("\n" + "=" * 60)
print("GRAPH-READY DATASET")
print("=" * 60)

print("Records :", len(graph_df))
print("Columns :", len(graph_df.columns))

print("\nMissing values:")
print(graph_df.isna().sum())

print("\nZones:")
print(graph_df["railwayzone"].value_counts())

print("\nJunction completeness:")
print(
    "Complete:",
    (
        graph_df["fromjunction"].notna() &
        graph_df["tojunction"].notna()
    ).sum()
)

print("\nGeometry:")
print(graph_df["st_lengthshape"].describe())

Initial records: 119340
After removing missing railwayzone: 116738
After removing incomplete junctions: 116218
After removing invalid geometry: 116218
After removing exact duplicates: 116218

GRAPH-READY DATASET
Records : 116218
Columns : 14

Missing values:
objectid               0
id                  1923
fromjunction           0
tojunction             0
speed              40966
type                 243
noofstations         225
tracklength          225
railwayzone            0
nooflanes           1230
bridge_yn              1
tunnel_yn            353
iso_countrycode        0
st_lengthshape         0
dtype: int64

Zones:
railwayzone
South Central Railway         15071
Western Railway               14193
South Western Railway          8475
Southern Railway               8004
North Western Railway          7705
Northern Railway               6857
West Central Railway           6800
East Coast Railway             6539
East Central Railway           6335
Northeast Frontier Railway     623

<h2>Graph EDA</h2>

In [184]:
# ===============================================================
# GRAPH EDA — STEP 1: NODE / EDGE STRUCTURE
# ===============================================================

print("=" * 60)
print("GRAPH STRUCTURE")
print("=" * 60)

# ---------------------------------------------------------------
# 1. UNIQUE NODES
# ---------------------------------------------------------------

from_nodes = graph_df["fromjunction"].dropna()
to_nodes = graph_df["tojunction"].dropna()

all_nodes = pd.concat([
    from_nodes,
    to_nodes
]).unique()

print("\n--- NODES ---")

print("Unique from-junctions :", from_nodes.nunique())
print("Unique to-junctions   :", to_nodes.nunique())
print("Unique nodes overall  :", len(all_nodes))


# ---------------------------------------------------------------
# 2. UNIQUE DIRECTED EDGES
# ---------------------------------------------------------------

edges = graph_df[
    ["fromjunction", "tojunction"]
].drop_duplicates()

print("\n--- EDGES ---")

print("Total graph records       :", len(graph_df))
print("Unique directed edges     :", len(edges))

print(
    "Multiple-record edge rows :",
    len(graph_df) - len(edges)
)


# ---------------------------------------------------------------
# 3. EDGE FREQUENCY
# ---------------------------------------------------------------

edge_frequency = (
    graph_df
    .groupby(
        ["fromjunction", "tojunction"]
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        "record_count",
        ascending=False
    )
)

print("\n--- EDGE FREQUENCY ---")

print(
    edge_frequency["record_count"].describe()
)

print("\nTop 20 repeated edges:")
print(
    edge_frequency.head(20).to_string(index=False)
)


# ---------------------------------------------------------------
# 4. SELF-LOOPS
# ---------------------------------------------------------------

self_loops = graph_df[
    graph_df["fromjunction"] ==
    graph_df["tojunction"]
]

print("\n--- SELF LOOPS ---")

print("Self-loop records :", len(self_loops))

print(
    "Unique self-loops :",
    self_loops[
        ["fromjunction", "tojunction"]
    ].drop_duplicates().shape[0]
)


# ---------------------------------------------------------------
# 5. BIDIRECTIONAL CONNECTIONS
# ---------------------------------------------------------------

directed_edges = set(
    zip(
        graph_df["fromjunction"],
        graph_df["tojunction"]
    )
)

bidirectional_pairs = set()

for u, v in directed_edges:

    if (v, u) in directed_edges:

        # Store in canonical order so A-B and B-A
        # count as one bidirectional connection.
        pair = tuple(sorted([u, v]))

        bidirectional_pairs.add(pair)

print("\n--- BIDIRECTIONAL CONNECTIONS ---")

print(
    "Bidirectional junction pairs:",
    len(bidirectional_pairs)
)


# ---------------------------------------------------------------
# 6. NODE DEGREE — DIRECTED
# ---------------------------------------------------------------

out_degree = (
    graph_df
    .groupby("fromjunction")
    .size()
    .sort_values(ascending=False)
)

in_degree = (
    graph_df
    .groupby("tojunction")
    .size()
    .sort_values(ascending=False)
)

print("\n--- NODE DEGREE ---")

print("\nHighest OUT-degree nodes:")
print(out_degree.head(15))

print("\nHighest IN-degree nodes:")
print(in_degree.head(15))


# ---------------------------------------------------------------
# 7. NODE DEGREE — UNIQUE CONNECTIONS
# ---------------------------------------------------------------

unique_out_degree = (
    graph_df
    .groupby("fromjunction")["tojunction"]
    .nunique()
    .sort_values(ascending=False)
)

unique_in_degree = (
    graph_df
    .groupby("tojunction")["fromjunction"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n--- UNIQUE NEIGHBOURS ---")

print("\nHighest unique OUT-neighbours:")
print(unique_out_degree.head(15))

print("\nHighest unique IN-neighbours:")
print(unique_in_degree.head(15))

GRAPH STRUCTURE

--- NODES ---
Unique from-junctions : 1535
Unique to-junctions   : 1646
Unique nodes overall  : 2024

--- EDGES ---
Total graph records       : 116218
Unique directed edges     : 2257
Multiple-record edge rows : 113961

--- EDGE FREQUENCY ---
count    2257.000000
mean       51.492246
std        85.626799
min         1.000000
25%         4.000000
50%        19.000000
75%        59.000000
max       711.000000
Name: record_count, dtype: float64

Top 20 repeated edges:
        fromjunction          tojunction  record_count
    Madgaon Junction              Thokur           711
Khurda Road Junction    Naupada Junction           694
Dharmavaram Junction     Pakala Junction           681
Himmatnagar Junction     Jai Samand Road           653
         Jammu Tawi             Baramula           643
   Udhna Chord Cabin    Jalgaon Junction           611
    Mudkhed Junction  Majri Bridge Cabin           610
 Peddapalli Junction  Nizamabad Junction           584
      Botad Juncti

In [185]:
# ===============================================================
# STEP 1A — BUILD CONCEPTUAL RAILWAY EDGES
# ===============================================================

edge_profile = (
    graph_df
    .groupby(
        ["fromjunction", "tojunction"],
        sort=False
    )
    .agg(
        record_count=("objectid", "size"),
        geometry_length_m=("st_lengthshape", "sum")
    )
    .reset_index()
)

print("Raw network records :", len(graph_df))
print("Unique graph edges  :", len(edge_profile))

print("\n--- RECORDS PER EDGE ---")
print(edge_profile["record_count"].describe())

print("\n--- TOP 20 EDGES ---")
print(
    edge_profile
    .sort_values("record_count", ascending=False)
    .head(20)
    .to_string(index=False)
)

Raw network records : 116218
Unique graph edges  : 2257

--- RECORDS PER EDGE ---
count    2257.000000
mean       51.492246
std        85.626799
min         1.000000
25%         4.000000
50%        19.000000
75%        59.000000
max       711.000000
Name: record_count, dtype: float64

--- TOP 20 EDGES ---
        fromjunction          tojunction  record_count  geometry_length_m
    Madgaon Junction              Thokur           711      306514.430567
Khurda Road Junction    Naupada Junction           694      261887.436300
Dharmavaram Junction     Pakala Junction           681      234517.063207
Himmatnagar Junction     Jai Samand Road           653      159284.320839
         Jammu Tawi             Baramula           643      390512.125304
   Udhna Chord Cabin    Jalgaon Junction           611      325489.351364
    Mudkhed Junction  Majri Bridge Cabin           610      276599.368149
 Peddapalli Junction  Nizamabad Junction           584      185560.098579
      Botad Junction   Saba

In [186]:
# ===============================================================
# TASK 1B — ATTRIBUTE CONSISTENCY WITHIN CONCEPTUAL EDGES
# ===============================================================

attributes = [
    "speed",
    "type",
    "noofstations",
    "tracklength",
    "nooflanes",
    "bridge_yn",
    "tunnel_yn",
    "railwayzone"
]

results = []

for col in attributes:

    variation = (
        graph_df
        .groupby(["fromjunction", "tojunction"])[col]
        .nunique(dropna=True)
    )

    results.append({
        "attribute": col,
        "constant": (variation <= 1).sum(),
        "variable": (variation > 1).sum(),
        "max_unique_values": variation.max()
    })

consistency_df = pd.DataFrame(results)

print("=" * 70)
print("ATTRIBUTE CONSISTENCY — 2,257 CONCEPTUAL EDGES")
print("=" * 70)

print(consistency_df.to_string(index=False))

ATTRIBUTE CONSISTENCY — 2,257 CONCEPTUAL EDGES
   attribute  constant  variable  max_unique_values
       speed      2217        40                  3
        type      2242        15                  2
noofstations      2196        61                  3
 tracklength      2162        95                  3
   nooflanes      2142       115                  3
   bridge_yn       381      1876                  2
   tunnel_yn      2173        84                  2
 railwayzone      2178        79                  3


In [187]:
# ===============================================================
# TASK 1C — INSPECT VARIABLE EDGE ATTRIBUTES
# ===============================================================

check_cols = [
    "speed",
    "type",
    "noofstations",
    "tracklength",
    "nooflanes",
    "railwayzone"
]

for col in check_cols:

    variation = (
        graph_df
        .groupby(["fromjunction", "tojunction"])[col]
        .nunique(dropna=True)
    )

    variable_pairs = variation[variation > 1].index

    print("\n" + "=" * 70)
    print(f"{col.upper()} — VARIABLE EDGES: {len(variable_pairs)}")
    print("=" * 70)

    if len(variable_pairs) == 0:
        continue

    sample = (
        graph_df
        .set_index(["fromjunction", "tojunction"])
        .loc[list(variable_pairs)]
        [[col, "objectid", "st_lengthshape"]]
        .head(30)
    )

    print(sample.to_string())


SPEED — VARIABLE EDGES: 40
                               speed  objectid  st_lengthshape
fromjunction tojunction                                       
Balotra      Samdari Junction  100.0     76886     1444.852101
             Samdari Junction  100.0     76887       38.276992
             Samdari Junction  100.0     76893     1111.041603
             Samdari Junction  100.0     76894       26.011894
             Samdari Junction  100.0     76895     4546.255237
             Samdari Junction  100.0     76896       19.955092
             Samdari Junction  100.0     76897      392.125943
             Samdari Junction  100.0     76898       55.309744
             Samdari Junction  100.0     76899      294.740776
             Samdari Junction  100.0     76900       24.450273
             Samdari Junction  100.0     76901     1517.902045
             Samdari Junction  100.0     76902       28.617801
             Samdari Junction  100.0     76903     3768.045244
             Samdari Juncti

In [188]:
# ===============================================================
# TASK 1D — BUILD FINAL CONCEPTUAL RAIL EDGES
# ===============================================================

def mode_or_nan(s):
    s = s.dropna()
    if s.empty:
        return np.nan
    return s.mode().iloc[0]


def count_unique(s):
    return s.dropna().nunique()


rail_edges = (
    graph_df
    .groupby(
        ["fromjunction", "tojunction"],
        sort=False
    )
    .agg(
        record_count=("objectid", "size"),

        geometry_length_m=("st_lengthshape", "sum"),

        speed=("speed", mode_or_nan),

        gauge=("type", mode_or_nan),

        no_of_stations=("noofstations", mode_or_nan),

        track_length=("tracklength", mode_or_nan),

        no_of_lanes=("nooflanes", mode_or_nan),

        bridge_present=(
            "bridge_yn",
            lambda x: (x == "Y").any()
        ),

        tunnel_present=(
            "tunnel_yn",
            lambda x: (x == "Y").any()
        ),

        railway_zone=("railwayzone", mode_or_nan),

        zone_count=("railwayzone", count_unique)
    )
    .reset_index()
)

# ---------------------------------------------------------------
# Convert geometry length to km
# ---------------------------------------------------------------

rail_edges["geometry_length_km"] = (
    rail_edges["geometry_length_m"] / 1000
)

# ---------------------------------------------------------------
# Add edge ID
# ---------------------------------------------------------------

rail_edges.insert(
    0,
    "edge_id",
    range(1, len(rail_edges) + 1)
)

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------

print("=" * 70)
print("FINAL CONCEPTUAL RAIL EDGE TABLE")
print("=" * 70)

print("Raw GIS records       :", len(graph_df))
print("Conceptual edges      :", len(rail_edges))
print(
    "Records represented   :",
    rail_edges["record_count"].sum()
)

print("\nColumns:")
print(rail_edges.columns.tolist())

print("\nFirst 20 edges:")
print(
    rail_edges.head(20).to_string(index=False)
)

print("\nEdge length statistics (km):")
print(
    rail_edges["geometry_length_km"].describe()
)

print("\nEdges crossing multiple railway zones:")
print(
    (rail_edges["zone_count"] > 1).sum()
)

print("\nSelf-loop edges:")
print(
    (
        rail_edges["fromjunction"] ==
        rail_edges["tojunction"]
    ).sum()
)

print("\nDuplicate conceptual edges:")
print(
    rail_edges.duplicated(
        ["fromjunction", "tojunction"]
    ).sum()
)

FINAL CONCEPTUAL RAIL EDGE TABLE
Raw GIS records       : 116218
Conceptual edges      : 2257
Records represented   : 116218

Columns:
['edge_id', 'fromjunction', 'tojunction', 'record_count', 'geometry_length_m', 'speed', 'gauge', 'no_of_stations', 'track_length', 'no_of_lanes', 'bridge_present', 'tunnel_present', 'railway_zone', 'zone_count', 'geometry_length_km']

First 20 edges:
 edge_id               fromjunction                  tojunction  record_count  geometry_length_m  speed        gauge  no_of_stations  track_length  no_of_lanes  bridge_present  tunnel_present     railway_zone  zone_count  geometry_length_km
       1       Tirunelveli Junction            Tenkasi Junction           177       71730.335937  110.0  Broad Gauge            16.0          72.0          1.0            True           False Southern Railway           1           71.730336
       2 Thiruvananthapuram Central      Nagercoil Bypass Cabin           107       71921.216716   80.0  Broad Gauge            31.0 

In [189]:
# ===============================================================
# TASK 2 — BUILD JUNCTION NODE TABLE
# ===============================================================

from_nodes = rail_edges["fromjunction"]
to_nodes = rail_edges["tojunction"]

all_nodes = pd.unique(
    pd.concat(
        [from_nodes, to_nodes],
        ignore_index=True
    )
)

nodes = pd.DataFrame({
    "node_id": range(1, len(all_nodes) + 1),
    "junction_name": all_nodes
})

# ---------------------------------------------------------------
# Degree information
# ---------------------------------------------------------------

out_degree = (
    rail_edges
    .groupby("fromjunction")
    .size()
    .rename("out_degree")
)

in_degree = (
    rail_edges
    .groupby("tojunction")
    .size()
    .rename("in_degree")
)

unique_out = (
    rail_edges
    .groupby("fromjunction")["tojunction"]
    .nunique()
    .rename("unique_out_neighbours")
)

unique_in = (
    rail_edges
    .groupby("tojunction")["fromjunction"]
    .nunique()
    .rename("unique_in_neighbours")
)

nodes = (
    nodes
    .set_index("junction_name")
    .join(out_degree)
    .join(in_degree)
    .join(unique_out)
    .join(unique_in)
    .fillna(0)
    .reset_index()
)

# Total directed degree
nodes["total_degree"] = (
    nodes["out_degree"] +
    nodes["in_degree"]
)

# Unique neighbours overall
nodes["unique_neighbours"] = (
    nodes["unique_out_neighbours"] +
    nodes["unique_in_neighbours"]
)

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------

print("=" * 70)
print("JUNCTION NODE TABLE")
print("=" * 70)

print("Total nodes:", len(nodes))
print("Total edges:", len(rail_edges))

print("\nDegree statistics:")
print(
    nodes[
        [
            "out_degree",
            "in_degree",
            "total_degree",
            "unique_neighbours"
        ]
    ].describe()
)

print("\nTop 20 nodes by total degree:")

print(
    nodes
    .sort_values("total_degree", ascending=False)
    .head(20)
    [
        [
            "junction_name",
            "out_degree",
            "in_degree",
            "total_degree",
            "unique_neighbours"
        ]
    ]
    .to_string(index=False)
)

print("\nNodes with degree = 0:")
print(
    (nodes["total_degree"] == 0).sum()
)

JUNCTION NODE TABLE
Total nodes: 2024
Total edges: 2257

Degree statistics:
        out_degree    in_degree  total_degree  unique_neighbours
count  2024.000000  2024.000000   2024.000000        2024.000000
mean      1.115119     1.115119      2.230237           2.230237
std       0.880440     0.781704      1.062717           1.062717
min       0.000000     0.000000      1.000000           1.000000
25%       1.000000     1.000000      1.000000           1.000000
50%       1.000000     1.000000      2.000000           2.000000
75%       2.000000     1.000000      3.000000           3.000000
max       6.000000     4.000000      7.000000           7.000000

Top 20 nodes by total degree:
           junction_name  out_degree  in_degree  total_degree  unique_neighbours
         Panvel Junction         5.0        2.0           7.0                7.0
       Bathinda Junction         4.0        2.0           6.0                6.0
        Mathura Junction         3.0        3.0           6.0    

In [190]:
# ===============================================================
# TASK 3 — PHYSICAL NETWORK CONNECTIVITY
# ===============================================================

import networkx as nx

# ---------------------------------------------------------------
# Build UNDIRECTED physical graph
# ---------------------------------------------------------------

G = nx.Graph()

# Add edges
G.add_edges_from(
    rail_edges[
        ["fromjunction", "tojunction"]
    ].itertuples(index=False, name=None)
)

print("=" * 70)
print("PHYSICAL RAILWAY NETWORK CONNECTIVITY")
print("=" * 70)

print("\nNodes :", G.number_of_nodes())
print("Edges :", G.number_of_edges())


# ---------------------------------------------------------------
# Connected components
# ---------------------------------------------------------------

components = list(nx.connected_components(G))

component_sizes = sorted(
    [len(c) for c in components],
    reverse=True
)

print("\n--- CONNECTED COMPONENTS ---")

print("Number of components:", len(components))

print("\nComponent size statistics:")
print(
    pd.Series(component_sizes).describe()
)

print("\nTop 20 component sizes:")

for i, size in enumerate(component_sizes[:20], 1):
    print(f"Component {i:2d}: {size:4d} nodes")


# ---------------------------------------------------------------
# Assign component ID to every node
# ---------------------------------------------------------------

component_map = {}

for component_id, component in enumerate(
    sorted(
        components,
        key=len,
        reverse=True
    ),
    start=1
):

    for node in component:
        component_map[node] = component_id


nodes["component_id"] = (
    nodes["junction_name"]
    .map(component_map)
)


# ---------------------------------------------------------------
# Component edge counts
# ---------------------------------------------------------------

component_edge_counts = []

for component_id, component in enumerate(
    sorted(
        components,
        key=len,
        reverse=True
    ),
    start=1
):

    subgraph = G.subgraph(component)

    component_edge_counts.append({
        "component_id": component_id,
        "nodes": subgraph.number_of_nodes(),
        "edges": subgraph.number_of_edges()
    })

component_summary = pd.DataFrame(
    component_edge_counts
)

print("\n--- COMPONENT SUMMARY ---")

print(
    component_summary.head(20).to_string(index=False)
)


# ---------------------------------------------------------------
# Main component coverage
# ---------------------------------------------------------------

largest_component = max(
    components,
    key=len
)

print("\n--- MAIN COMPONENT ---")

print(
    "Largest component nodes:",
    len(largest_component)
)

print(
    "Largest component edges:",
    G.subgraph(largest_component).number_of_edges()
)

print(
    "Percentage of nodes in largest component:",
    round(
        len(largest_component) /
        G.number_of_nodes() * 100,
        2
    ),
    "%"
)

print(
    "Percentage of edges in largest component:",
    round(
        G.subgraph(largest_component).number_of_edges() /
        G.number_of_edges() * 100,
        2
    ),
    "%"
)


# ---------------------------------------------------------------
# Small components
# ---------------------------------------------------------------

small_components = [
    c for c in components
    if len(c) <= 5
]

print("\nComponents with <= 5 nodes:",
      len(small_components))

if small_components:

    print("\nSmall components:")

    for component in sorted(
        small_components,
        key=len
    )[:20]:

        print(
            list(component)
        )

PHYSICAL RAILWAY NETWORK CONNECTIVITY

Nodes : 2024
Edges : 2249

--- CONNECTED COMPONENTS ---
Number of components: 113

Component size statistics:
count     113.000000
mean       17.911504
std       165.157909
min         2.000000
25%         2.000000
50%         2.000000
75%         2.000000
max      1758.000000
dtype: float64

Top 20 component sizes:
Component  1: 1758 nodes
Component  2:    8 nodes
Component  3:    6 nodes
Component  4:    5 nodes
Component  5:    5 nodes
Component  6:    5 nodes
Component  7:    4 nodes
Component  8:    4 nodes
Component  9:    4 nodes
Component 10:    4 nodes
Component 11:    4 nodes
Component 12:    3 nodes
Component 13:    3 nodes
Component 14:    3 nodes
Component 15:    3 nodes
Component 16:    3 nodes
Component 17:    3 nodes
Component 18:    3 nodes
Component 19:    3 nodes
Component 20:    3 nodes

--- COMPONENT SUMMARY ---
 component_id  nodes  edges
            1   1758   2093
            2      8      7
            3      6      5
    

In [191]:
# ===============================================================
# TASK 3B — COMPONENT CLASSIFICATION
# ===============================================================

# Create a lookup for component membership
component_lookup = {}

for component_id, component in enumerate(
    sorted(components, key=len, reverse=True),
    start=1
):
    for node in component:
        component_lookup[node] = component_id


# ---------------------------------------------------------------
# Build component-level edge information
# ---------------------------------------------------------------

component_edges = []

for component_id, component in enumerate(
    sorted(components, key=len, reverse=True),
    start=1
):

    component_set = set(component)

    mask = (
        rail_edges["fromjunction"].isin(component_set) &
        rail_edges["tojunction"].isin(component_set)
    )

    sub_edges = rail_edges[mask]

    component_edges.append({
        "component_id": component_id,
        "nodes": len(component),
        "edges": len(sub_edges),
        "total_length_km": sub_edges["geometry_length_km"].sum(),
        "zones": sub_edges["railway_zone"].nunique(),
        "self_loops": (
            sub_edges["fromjunction"] ==
            sub_edges["tojunction"]
        ).sum()
    })


component_summary = pd.DataFrame(component_edges)


# ---------------------------------------------------------------
# Display all components except the main component
# ---------------------------------------------------------------

print("=" * 70)
print("NON-MAIN CONNECTED COMPONENTS")
print("=" * 70)

print(
    component_summary[
        component_summary["component_id"] != 1
    ]
    .to_string(index=False)
)


# ---------------------------------------------------------------
# Show node names for components larger than 1 node
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPONENT NODE MEMBERS")
print("=" * 70)

for component_id, component in enumerate(
    sorted(components, key=len, reverse=True)[1:],
    start=2
):

    if len(component) <= 10:

        print(
            f"\nComponent {component_id} "
            f"({len(component)} nodes):"
        )

        print(
            sorted(component)
        )

NON-MAIN CONNECTED COMPONENTS
 component_id  nodes  edges  total_length_km  zones  self_loops
            2      8      7       141.504245      1           0
            3      6      5       280.638516      2           0
            4      5      4        92.021844      1           0
            5      5      4       145.327744      1           0
            6      5      5       111.021903      1           0
            7      4      3        80.137136      1           0
            8      4      3        28.113571      1           0
            9      4      3        98.776045      1           0
           10      4      3        60.840900      1           0
           11      4      3        86.192884      1           0
           12      3      2        36.453884      1           0
           13      3      2        29.372587      1           0
           14      3      2        56.868105      1           0
           15      3      2        39.772213      1           0
          

In [192]:
# ===============================================================
# TASK 4 — JUNCTION NAME NORMALIZATION CHECK
# ===============================================================

import re

def normalize_junction_name(x):
    if pd.isna(x):
        return x

    x = str(x)

    # whitespace normalization
    x = re.sub(r"\s+", " ", x).strip()

    # normalize obvious punctuation spacing
    x = re.sub(r"\s+([.,)])", r"\1", x)
    x = re.sub(r"([(])\s+", r"\1", x)

    return x


nodes["normalized_name"] = (
    nodes["junction_name"]
    .apply(normalize_junction_name)
)

# ---------------------------------------------------------------
# Names that collapse after normalization
# ---------------------------------------------------------------

name_groups = (
    nodes
    .groupby("normalized_name")
    .agg(
        original_count=("junction_name", "nunique"),
        original_names=("junction_name", list),
        component_count=("component_id", "nunique")
    )
    .reset_index()
)

name_conflicts = name_groups[
    name_groups["original_count"] > 1
].sort_values(
    ["component_count", "original_count"],
    ascending=False
)

print("=" * 70)
print("JUNCTION NAME NORMALIZATION")
print("=" * 70)

print(
    "Original junction nodes:",
    len(nodes)
)

print(
    "Normalized unique names:",
    nodes["normalized_name"].nunique()
)

print(
    "Potential duplicate-name groups:",
    len(name_conflicts)
)

print("\n--- NORMALIZATION GROUPS ---")

print(
    name_conflicts.head(50).to_string(index=False)
)

# ---------------------------------------------------------------
# Groups that occur across multiple components
# ---------------------------------------------------------------

component_split_candidates = name_conflicts[
    name_conflicts["component_count"] > 1
]

print("\n" + "=" * 70)
print("NAMES SPLIT ACROSS MULTIPLE COMPONENTS")
print("=" * 70)

print(
    "Potential component-splitting names:",
    len(component_split_candidates)
)

print(
    component_split_candidates.to_string(index=False)
)

JUNCTION NAME NORMALIZATION
Original junction nodes: 2024
Normalized unique names: 1987
Potential duplicate-name groups: 36

--- NORMALIZATION GROUPS ---
              normalized_name  original_count                                                     original_names  component_count
              Bandel Junction               3              [Bandel Junction, Bandel  Junction, Bandel Junction ]                2
                      Algapur               2                                                [Algapur , Algapur]                2
                     Antagarh               2                                             [Antagarh, Antagarh\n]                2
Bhadrachalam Road Chord Cabin               2 [Bhadrachalam Road Chord Cabin, Bhadrachalam Road Chord Cabin\r\n]                2
                        Dadri               2                                                    [Dadri , Dadri]                2
                      Hortoki               2                     

In [193]:
# ===============================================================
# TASK 5 — STATION DATASET STRUCTURE
# ===============================================================

print("=" * 70)
print("STATION DATASET — STRUCTURE CHECK")
print("=" * 70)

print("Shape:", stations_df.shape)

print("\nColumns:")
print(stations_df.columns.tolist())

print("\nMissing values:")
print(
    stations_df.isna().sum()
)

print("\nUnique values:")
print(
    stations_df.nunique()
)

print("\nFirst 10 rows:")
print(
    stations_df.head(10).to_string(index=False)
)

STATION DATASET — STRUCTURE CHECK


NameError: name 'stations_df' is not defined

In [194]:
# ===============================================================
# SAVE — INITIAL RAILWAY PHYSICAL GRAPH
# ===============================================================

# Save conceptual edge table
rail_edges.to_csv(
    "rail_edges_initial.csv",
    index=False
)

# Save junction node table
nodes.to_csv(
    "junction_nodes_initial.csv",
    index=False
)

# Save component information
component_summary.to_csv(
    "railway_components_initial.csv",
    index=False
)

print("=" * 70)
print("INITIAL RAILWAY GRAPH SAVED")
print("=" * 70)

print(f"Rail edges       : {len(rail_edges):,}")
print(f"Junction nodes   : {len(nodes):,}")
print(f"Components       : {len(component_summary):,}")

print("\nFiles:")
print("1. rail_edges_initial.csv")
print("2. junction_nodes_initial.csv")
print("3. railway_components_initial.csv")

INITIAL RAILWAY GRAPH SAVED
Rail edges       : 2,257
Junction nodes   : 2,024
Components       : 113

Files:
1. rail_edges_initial.csv
2. junction_nodes_initial.csv
3. railway_components_initial.csv
